# BTCUSDT time-to-high: independent 1-hour forecast screen
**New 1h-only research; no 15m results are required.** Upload this notebook to Colab and use a **separate CPU runtime** from the running 15m notebook. A second browser tab alone does not establish a separate runtime. Do not disconnect or restart your active 15m runtime. If a separate runtime is unavailable, run this notebook after the first one stops. Do not run two copies of this 1h notebook at once.

Quota-repair update: prefer one active horizon at a time for now. Separate output folders prevent file conflicts but do not isolate Google API quotas. Requests are now paced, and explicit rate-limit errors wait and retry. Model/runner hashes and checkpoint compatibility remain unchanged. If this notebook failed previously, restart its runtime and resume this updated version after a few minutes. Keep its existing checkpoints.

Source metadata compatibility: a hash-pinned execution patch permits only modifiedTime differences when file ID, name, size and MD5 still match. All other source changes stop execution. SOURCE_METADATA_COMPAT_1h.json records observed metadata and original/executed code hashes. The frozen contracts and saved cache/model identities remain unchanged.

Run all and authorize the same Google account. Two fit workers maximum. Four-hour soft session budget includes loading, then finishes active fits/loading month and saves progress. Rerun the same notebook to resume. A session may overrun during reads, active fits, uploads or report generation.

Same 386 features, five chronological validation periods, six inner settings, train-only preprocessing, strict 60-minute label-end purging, baseline evaluation rows and missing-data fallback policy. Reserved rows are excluded. No model/quality shortcuts. No completed 1h pilot fits exist to reuse.

**Parallel write isolation:** everything new goes under **Timing_1h_Run** inside the existing Drive output folder. Completed shared feature caches may be read, but this notebook never writes shared caches. Missing or incomplete shared caches are loaded into private 1h caches using authenticated remote ZIP ranges. This can duplicate some loading/cache storage while both horizons run, but prevents concurrent writes to the same files. No dataset is downloaded to your PC.

The first session also builds the 1h target cache from research partitions only. Expect loading messages before fitting. All original datasets and historical pilot receipts are read-only. Source/runtime/hash checks stop incompatible resumes.

Inside **Timing_1h_Run**:
- **SCREEN_1h_RESULTS.json**: session progress.
- **Timing_1h_Fit_Scores**: compact resumable fit checkpoints; no models or row predictions.
- **Timing_Research_Reports/60m_summary.pdf**: charts and coverage; 60m means 1h.
- **60m_feature_rankings.csv** and **60m_fold_details.csv** in the reports folder: all candidates and completed fold results.

Reports update after each family and at session end. Other-horizon counts show the frozen pilot only, not the live 15m run; the two writers do not merge reports during concurrent execution. These are provisional development results, not significance or trading evidence.

Keep the 15m notebook as it is. Send this notebook's final status and report-folder link when ready. This notebook does not depend on the 15m run finishing.


In [ ]:
import hashlib, json
sources = {'reference': '"""BTCUSDT timing feature screen: pilot by default, full scope only after review.\nColab supplies CONFIG, RUNNER_SHA256 and DriveRangeReader in the notebook.\n"""\nimport io\nimport json\nimport re\nimport hashlib\nimport zipfile\nimport struct\nimport zlib\nimport time\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport numpy as np\nimport pandas as pd\n\n\ndef sha(data): return hashlib.sha256(data).hexdigest()\ndef signature(obj): return sha(json.dumps(obj,sort_keys=True,allow_nan=False).encode())\ndef utc(value): return pd.Timestamp(value,tz=\'UTC\')\n\n\ndef partition_masks(times, horizon, first, last):\n    start,stop=utc(first),utc(last)\n    ends=times+pd.Timedelta(minutes=horizon)\n    inner_start=start-pd.DateOffset(months=3)\n    return {\'inner_train\':(times<inner_start)&(ends<inner_start),\n            \'inner_valid\':(times>=inner_start)&(times<start)&(ends<start),\n            \'outer_train\':(times<start)&(ends<start),\n            \'outer_valid\':(times>=start)&(times<stop)}\n\n\ndef preprocess(x_train,x_predict,minimum):\n    observed=np.isfinite(x_train)\n    if observed.sum()<minimum or np.unique(x_train[observed]).size<2:\n        return None\n    median=float(np.median(x_train[observed]))\n    def transform(x):\n        missing=~np.isfinite(x)\n        return np.column_stack([np.where(missing,median,x),missing.astype(float)]).astype(np.float32)\n    return transform(x_train),transform(x_predict),median\n\n\ndef manual_predict(model,x):\n    # Independent traversal checks saved sklearn leaf predictions without another fit.\n    tree=model.tree_;nodes=np.zeros(len(x),dtype=np.int64)\n    for _ in range(model.get_depth()+1):\n        active=tree.children_left[nodes]!=-1\n        if not active.any():break\n        rows=np.flatnonzero(active);current=nodes[rows]\n        left=x[rows,tree.feature[current]]<=tree.threshold[current]\n        nodes[rows]=np.where(left,tree.children_left[current],tree.children_right[current])\n    assert (tree.children_left[nodes]==-1).all()\n    return tree.value[nodes,0,0]\n\n\ndef tree_fit(x_train,y_train,x_valid,y_valid,depth,leaf,minimum):\n    from sklearn.tree import DecisionTreeRegressor\n    start=time.perf_counter();prepared=preprocess(x_train,x_valid,minimum)\n    coverage={\'train_observed\':int(np.isfinite(x_train).sum()),\'train_rows\':len(x_train),\n              \'validation_observed\':int(np.isfinite(x_valid).sum()),\'validation_rows\':len(x_valid)}\n    if prepared is None:return {\'status\':\'INSUFFICIENT_TRAINING_COVERAGE_OR_VARIATION\',\'fit_count\':0,\'coverage\':coverage}\n    train,valid,median=prepared\n    model=DecisionTreeRegressor(criterion=\'absolute_error\',max_depth=depth,min_samples_leaf=leaf,random_state=48)\n    model.fit(train,y_train);prediction=model.predict(valid)\n    assert np.allclose(prediction,manual_predict(model,valid),rtol=0,atol=1e-10)\n    error=np.abs(prediction-y_valid);observed=np.isfinite(x_valid)\n    return {\'status\':\'PASS\',\'fit_count\':1,\'depth\':depth,\'min_samples_leaf\':leaf,\n            \'MAE_minutes\':float(error.mean()),\'absolute_error_sum\':float(error.sum()),\n            \'observed_rows_MAE\':float(error[observed].mean()) if observed.any() else None,\n            \'missing_rows_MAE\':float(error[~observed].mean()) if (~observed).any() else None,\n            \'coverage\':coverage,\'imputation_median\':median,\'tree_nodes\':int(model.tree_.node_count),\n            \'seconds\':time.perf_counter()-start,\'independent_tree_traversal\':\'PASS\'}\n\n\ndef pick_setting(results):\n    valid=[v for v in results if v[\'status\']==\'PASS\']\n    return min(valid,key=lambda v:(v[\'MAE_minutes\'],v[\'depth\'],-v[\'min_samples_leaf\'])) if valid else None\n\n\ndef self_test():\n    times=pd.date_range(\'2022-10-01\',\'2023-04-02\',freq=\'30min\',tz=\'UTC\')\n    masks=partition_masks(times,18000,\'2023-04-01\',\'2023-07-01\')\n    ends=times+pd.Timedelta(minutes=18000)\n    assert (ends[masks[\'inner_train\']]<utc(\'2023-01-01\')).all()\n    assert (ends[masks[\'inner_valid\']]<utc(\'2023-04-01\')).all()\n    assert (ends[masks[\'outer_train\']]<utc(\'2023-04-01\')).all()\n    prep=preprocess(np.array([0.,2.,np.nan]),np.array([10000.,np.nan]),2)\n    assert prep[2]==1 and prep[1][1,0]==1 and prep[1][1,1]==1\n    assert preprocess(np.zeros(10),np.ones(2),2) is None\n    x=np.repeat([0.,1.,2.,3.],100);y=np.where(x<2,1.,9.)\n    for depth,leaf in [(2,64),(3,64),(5,256)]:\n        score=tree_fit(x,y,np.array([0.,3.,np.nan]),np.array([1.,9.,1.]),depth,leaf,100)\n        assert score[\'status\']==\'PASS\' and score[\'coverage\'][\'validation_rows\']==3\n    tied=[{\'status\':\'PASS\',\'MAE_minutes\':1,\'depth\':3,\'min_samples_leaf\':64},\n          {\'status\':\'PASS\',\'MAE_minutes\':1,\'depth\':2,\'min_samples_leaf\':64},\n          {\'status\':\'PASS\',\'MAE_minutes\':1,\'depth\':2,\'min_samples_leaf\':256}]\n    assert pick_setting(tied)[\'min_samples_leaf\']==256\n    print(\'PASS: temporal purges, train-only imputation, missing-row preservation, deterministic ties and tree replay.\',flush=True)\n\n\nclass DriveStore:\n    def __init__(self,service,folder):self.service,self.folder=service,folder;self.ids={}\n    def find(self,name):\n        if name not in self.ids:\n            files=self.service.files().list(q=f"\'{self.folder}\' in parents and name=\'{name}\' and trashed=false",fields=\'files(id)\').execute()[\'files\']\n            assert len(files)<=1,\'Ambiguous output: \'+name\n            self.ids[name]=files[0][\'id\'] if files else None\n        return self.ids[name]\n    def read(self,name):\n        file=self.find(name)\n        return self.service.files().get_media(fileId=file).execute() if file else None\n    def write(self,name,data,mime=\'application/json\'):\n        from googleapiclient.http import MediaIoBaseUpload\n        assert len(data)<=128*1024**2,\'Compact cache limit exceeded\'\n        media=MediaIoBaseUpload(io.BytesIO(data),mimetype=mime,resumable=len(data)>5*1024**2)\n        file=self.find(name)\n        request=self.service.files().update(fileId=file,media_body=media,fields=\'id\') if file else self.service.files().create(body={\'name\':name,\'parents\':[self.folder]},media_body=media,fields=\'id\')\n        self.ids[name]=request.execute()[\'id\']\n    def json(self,name,obj):self.write(name,json.dumps(obj,indent=2,allow_nan=False).encode())\n\n\nclass RemoteArchive:\n    """Index once, read independent compressed members with bounded parallel I/O."""\n    def __init__(self,credentials,meta):\n        from google.auth.transport.requests import AuthorizedSession\n        self.credentials,self.meta=credentials,meta;self.local=threading.local();self.lock=threading.Lock();self.bytes=0\n        reader=DriveRangeReader(AuthorizedSession(credentials),meta[\'id\'],int(meta[\'size\']))\n        with zipfile.ZipFile(reader) as archive:self.members={x.filename:x for x in archive.infolist()}\n        self.bytes=reader.transferred\n    def read_range(self,start,length):\n        from google.auth.transport.requests import AuthorizedSession\n        if not hasattr(self.local,\'session\'):self.local.session=AuthorizedSession(self.credentials)\n        with self.lock:\n            assert self.bytes+length<=12*1024**3,\'Remote session byte budget exceeded\'\n            self.bytes+=length\n        reader=DriveRangeReader(self.local.session,self.meta[\'id\'],int(self.meta[\'size\']));reader.seek(start)\n        return reader.read(length)\n    def read(self,name):\n        info=self.members[name];header=self.read_range(info.header_offset,30)\n        fields=struct.unpack(\'<4s5H3I2H\',header);assert fields[0]==b\'PK\\x03\\x04\' and not fields[2]&1\n        raw=self.read_range(info.header_offset+30+fields[-2]+fields[-1],info.compress_size)\n        if info.compress_type==zipfile.ZIP_DEFLATED:data=zlib.decompress(raw,-15)\n        elif info.compress_type==zipfile.ZIP_STORED:data=raw\n        else:raise RuntimeError(\'Unsupported compression method; no full download fallback\')\n        assert len(data)==info.file_size and zlib.crc32(data)&0xffffffff==info.CRC\n        return data\n\n\ndef read_cache(store,name,expected):\n    raw=store.read(name)\n    if raw is None:return {\'identity\':expected,\'completed\':[],\'member_hashes\':{}},None,None\n    with np.load(io.BytesIO(raw),allow_pickle=False) as cache:\n        meta=json.loads(str(cache[\'metadata\'].item()));assert meta[\'identity\']==expected,\'Cache identity changed: \'+name\n        times=cache[\'times\'];values=cache[\'values\']\n        assert meta[\'array_sha256\']==sha(times.tobytes()+values.tobytes()),\'Cache content hash failed\'\n    return meta,times,values\n\n\ndef write_cache(store,name,meta,times,values):\n    meta[\'array_sha256\']=sha(times.tobytes()+values.tobytes())\n    buf=io.BytesIO();np.savez_compressed(buf,metadata=np.array(json.dumps(meta)),times=times,values=values)\n    store.write(name,buf.getvalue(),\'application/octet-stream\')\n\n\ndef load_target(archive,store,root,manifest,horizon,source,config):\n    import pyarrow.parquet as pq\n    name=f\'SCREEN_Y_{horizon}m.npz\';identity={\'type\':\'valid_time_high_research_v1\',\'source\':source,\'horizon\':horizon,\'origin_end\':config[\'origin_end_exclusive\'],\'cadence\':30}\n    meta,times,values=read_cache(store,name,identity)\n    ts=[] if times is None else [times];ys=[] if values is None else [values]\n    for month in config[\'months\']:\n        if month in meta[\'completed\']:continue\n        member=root+f\'y/horizon_minutes={horizon}/symbol=BTCUSDT/source_month={month}/part.parquet\'\n        raw=archive.read(member);assert sha(raw)==manifest[member]\n        frame=pq.read_table(io.BytesIO(raw),columns=[\'canonical_symbol\',\'decision_time\',\'horizon_minutes\',\'time_to_high_min\',\'label_status\']).to_pandas()\n        t=pd.DatetimeIndex(pd.to_datetime(frame.decision_time,utc=True));assert t.is_unique and frame.canonical_symbol.eq(\'BTCUSDT\').all() and frame.horizon_minutes.eq(horizon).all()\n        keep=(t.asi8%(30*60*10**9)==0)&(t< pd.Timestamp(config[\'origin_end_exclusive\']))&frame.label_status.eq(\'VALID\').to_numpy()\n        t=t[keep];y=frame.loc[keep,\'time_to_high_min\'].to_numpy(dtype=float)\n        assert np.isfinite(y).all() and ((y>=0)&(y<=horizon)).all()\n        assert ((t+pd.Timedelta(minutes=horizon))<pd.Timestamp(config[\'reserve_start\'])).all()\n        ts.append(t.asi8);ys.append(y);meta[\'completed\'].append(month);meta[\'member_hashes\'][member]=manifest[member]\n        write_cache(store,name,meta,np.concatenate(ts),np.concatenate(ys))\n        print(f\'Y {horizon}m: cached research month {month}\',flush=True)\n    times=np.concatenate(ts);values=np.concatenate(ys)\n    assert len(times)==len(np.unique(times)) and (np.diff(times)>0).all()\n    return pd.DatetimeIndex(times,tz=\'UTC\'),values,meta[\'array_sha256\']\n\n\ndef load_family(archive,store,family,columns,source,checksums,config,workers):\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    name=f\'SCREEN_X_F4Z_{family}.npz\'\n    identity={\'type\':\'f4z_research_cache_v1\',\'source\':source,\'columns\':columns,\'origin_end\':config[\'origin_end_exclusive\'],\n              \'cadence\':30,\'minimum_baseline_n\':1000,\'allowed_source_status\':[\'PROVISIONAL\']}\n    meta,times,values=read_cache(store,name,identity)\n    ts=[] if times is None else [times];xs=[] if values is None else [values]\n    members={}\n    for member in archive.members:\n        match=re.search(r\'/dates/[^/=]+=(\\d{4}-\\d{2}-\\d{2})/\',member)\n        if match and \'/F4Z/\' in member and f\'/family={family}/\' in member and \'/symbol=BTCUSDT/\' in member and member.endswith(\'.parquet\'):\n            if \'2022-04-22\'<=match[1]<=\'2024-06-30\':assert match[1] not in members;members[match[1]]=member\n    fields=[c.split(\'__\',2)[2] for c in columns]\n    wanted=[\'decision_time\',\'canonical_symbol\',\'available_after\']\n    for field in fields:wanted += [field,field[:-7]+\'__valid\',field[:-7]+\'__baseline_n\',field[:-7]+\'__scientific_status\']\n    def read_day(day):\n        member=members[day];raw=archive.read(member);assert sha(raw)==checksums[member]\n        table=pq.read_table(io.BytesIO(raw),columns=wanted)\n        t=pd.DatetimeIndex(pd.to_datetime(table[\'decision_time\'].to_pandas(),utc=True))\n        expected=pd.date_range(utc(day)+pd.Timedelta(minutes=1),periods=1440,freq=\'min\')\n        assert t.equals(expected) and set(table[\'canonical_symbol\'].to_pylist())=={\'BTCUSDT\'}\n        keep=(t.asi8%(30*60*10**9)==0)&(t<pd.Timestamp(config[\'origin_end_exclusive\']))\n        table=table.take(pa.array(np.flatnonzero(keep)));t=t[keep]\n        available=pd.DatetimeIndex(pd.to_datetime(table[\'available_after\'].to_pandas(),utc=True));causal=~available.isna()&(available<=t)\n        data=np.empty((len(t),len(fields)),dtype=np.float64)\n        for i,field in enumerate(fields):\n            base=field[:-7];v=table[field].cast(pa.float64()).to_numpy(zero_copy_only=False)\n            valid=table[base+\'__valid\'].to_pandas().fillna(False).to_numpy(dtype=bool)\n            n=table[base+\'__baseline_n\'].cast(pa.float64()).to_numpy(zero_copy_only=False)\n            status=np.array(table[base+\'__scientific_status\'].to_pylist())\n            good=causal&valid&np.isfinite(v)&np.isfinite(n)&(n>=1000)&(status==\'PROVISIONAL\')\n            data[:,i]=np.where(good,v,np.nan)\n        return t.asi8,data,member,checksums[member]\n    for month in config[\'months\']:\n        if month in meta[\'completed\']:continue\n        days=pd.date_range(max(utc(month+\'-01\'),utc(\'2022-04-22\')),utc(month+\'-01\')+pd.offsets.MonthEnd(0),freq=\'D\').strftime(\'%Y-%m-%d\').tolist()\n        assert set(days)<=set(members)\n        with ThreadPoolExecutor(max_workers=workers) as executor:\n            for count,(t,x,member,h) in enumerate(executor.map(read_day,days),1):\n                ts.append(t);xs.append(x);meta[\'member_hashes\'][member]=h\n                if count%5==0:print(f\'X {family} {month}: {count}/{len(days)} days read\',flush=True)\n        meta[\'completed\'].append(month);write_cache(store,name,meta,np.concatenate(ts),np.concatenate(xs))\n        print(f\'X {family}: cached research month {month}\',flush=True)\n    times=np.concatenate(ts);values=np.concatenate(xs)\n    assert len(times)==len(np.unique(times)) and (np.diff(times)>0).all()\n    return pd.DatetimeIndex(times,tz=\'UTC\'),values,meta[\'array_sha256\']\n\n\ndef check_baselines(times,y,horizon,baseline,config):\n    for i,(first,last) in enumerate(config[\'folds\']):\n        masks=partition_masks(times,horizon,first,last);train=masks[\'outer_train\'];valid=masks[\'outer_valid\']\n        expected=baseline[\'completed\'][str(horizon)][\'folds\'][i]\n        assert train.sum()==expected[\'train_rows\'] and valid.sum()==expected[\'validation_rows\']\n        constants=[np.median(y[train]),horizon/2,np.mean(y[train])]\n        for name,pred in zip([\'training_median_time_to_high\',\'horizon_midpoint\',\'training_mean_time_to_high\'],constants):\n            assert abs(float(np.abs(y[valid]-pred).mean())-expected[\'scores\'][name][\'MAE_minutes\'])<1e-8\n\n\ndef update_matrix(state,baseline,config):\n    for feature in config[\'candidate_features\']:\n        for horizon in config[\'horizons\']:\n            keys=[f\'{feature}|{horizon}|{i}\' for i in range(len(config[\'folds\']))]\n            if not all(key in state[\'folds\'] for key in keys):continue\n            folds=[state[\'folds\'][key] for key in keys]\n            count=sum(v[\'validation_rows\'] for v in folds);mae=sum(v[\'absolute_error_sum\'] for v in folds)/count\n            base=baseline[\'completed\'][str(horizon)][\'pooled\']\n            state[\'matrix\'][f\'{feature}|{horizon}\']={\'feature\':feature,\'horizon_minutes\':horizon,\'rows\':count,\'MAE_minutes\':mae,\n                \'MAE_percent_of_horizon\':100*mae/horizon,\'delta_MAE_vs_baselines\':{n:mae-v[\'MAE_minutes\'] for n,v in base.items()},\n                \'folds_using_tree\':sum(v[\'status\']==\'PASS\' for v in folds),\'folds_using_fallback\':sum(v[\'status\']!=\'PASS\' for v in folds),\n                \'status\':\'PROVISIONAL_DEVELOPMENT_SCREEN_NOT_TEST_PROMOTION\'}\n\n\ndef run(mode=\'pilot\'):\n    import re\n    import platform\n    import sklearn\n    import pyarrow\n    import os\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from threadpoolctl import threadpool_limits\n    assert mode in (\'pilot\',\'full\');self_test();auth.authenticate_user();credentials,_=google.auth.default()\n    service=build(\'drive\',\'v3\',credentials=credentials,cache_discovery=False);store=DriveStore(service,CONFIG[\'folder\'])\n    sources={k:service.files().get(fileId=v,fields=\'id,name,size,md5Checksum,modifiedTime\').execute() for k,v in CONFIG[\'source_ids\'].items()}\n    assert sources==CONFIG[\'audit_sources\'],\'Source archives changed since the feature audit\'\n    versions={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__,\'pyarrow\':pyarrow.__version__}\n    identity={\'config\':CONFIG,\'sources\':sources,\'runner_sha256\':RUNNER_SHA256,\'versions\':versions};sig=signature(identity)\n    prior=store.read(\'SCREEN_RESULTS.json\')\n    state=json.loads(prior) if prior else {\'signature\':sig,\'identity\':identity,\'jobs\':{},\'matrix\':{},\'status\':\'RUNNING_PILOT\',\'new_model_fits_total\':0,\'reserved_partitions_read\':0}\n    assert state[\'signature\']==sig,\'Screen identity changed; inspect results before replacement.\'\n    baseline=json.loads(store.read(\'BASELINE_RESULTS.json\'))\n    assert signature(baseline[\'identity\'])==baseline[\'signature\']==CONFIG[\'baseline_signature\']\n    assert baseline[\'identity\'][\'source\']==sources[\'Y\']\n    features=CONFIG[\'pilot_features\'] if mode==\'pilot\' else CONFIG[\'candidate_features\']\n    horizons=CONFIG[\'pilot_horizons\'] if mode==\'pilot\' else CONFIG[\'horizons\']\n    todo=[(f,h) for f in features for h in horizons if f\'{f}|{h}\' not in state[\'jobs\']]\n    if not todo:\n        state[\'status\']=\'PILOT_COMPLETE_REVIEW_BEFORE_FULL\' if mode==\'pilot\' else \'FULL_INDIVIDUAL_SCREEN_COMPLETE\'\n        store.json(\'SCREEN_RESULTS.json\',state)\n        print(\'Requested scope already complete. Reusing saved results; no archive reads.\');return\n    fit_workers=min(2,os.cpu_count() or 1);io_workers=min(4,os.cpu_count() or 1)\n    state[\'workers\']={\'fits\':fit_workers,\'remote_reads\':io_workers}\n    print(f\'Scope={mode}; {len(features)} features x {len(horizons)} horizons. Fits={fit_workers} workers; reads={io_workers} workers.\',flush=True)\n    folders=service.files().list(q=f"\'{CONFIG[\'folder\']}\' in parents and name=\'Timing_Screen_Fit_Scores\' and mimeType=\'application/vnd.google-apps.folder\' and trashed=false",fields=\'files(id)\').execute()[\'files\']\n    assert len(folders)<=1\n    fit_folder=folders[0][\'id\'] if folders else service.files().create(body={\'name\':\'Timing_Screen_Fit_Scores\',\'parents\':[CONFIG[\'folder\']],\'mimeType\':\'application/vnd.google-apps.folder\'},fields=\'id\').execute()[\'id\']\n    fit_store=DriveStore(service,fit_folder)\n    start=time.perf_counter();archives={k:RemoteArchive(credentials,v) for k,v in sources.items()}\n    yarchive=archives[\'Y\'];manifest_name=next(n for n in yarchive.members if n.endswith(\'/HANDOFF_2_MANIFEST.json\'))\n    root=manifest_name[:-len(\'HANDOFF_2_MANIFEST.json\')];manifest={root+e[\'path\']:e[\'sha256\'] for e in json.loads(yarchive.read(manifest_name))[\'files\']}\n    raw=archives[\'X\'].read(\'FILES.sha256\');checksums={}\n    for line in raw.decode().splitlines():\n        if line.strip():h,name=line.split(maxsplit=1);checksums[name.lstrip(\'*\')]=h\n    targets={}\n    target_start=time.perf_counter()\n    for horizon in sorted({h for f,h in todo}):\n        targets[horizon]=load_target(yarchive,store,root,manifest,horizon,sources[\'Y\'],CONFIG)\n        check_baselines(targets[horizon][0],targets[horizon][1],horizon,baseline,CONFIG)\n    state[\'target_loading_seconds_this_session\']=time.perf_counter()-target_start\n    def save():\n        state[\'session_elapsed_seconds\']=time.perf_counter()-start\n        state[\'remote_bytes_this_session\']={k:v.bytes for k,v in archives.items()}\n        state[\'new_model_fits_total\']=sum(v[\'fit_count\'] for v in state[\'jobs\'].values())\n        state[\'summed_fit_seconds\']=sum(v.get(\'fit_seconds\',0) for v in state[\'jobs\'].values())\n        try:\n            import resource\n            state[\'peak_process_RSS_MiB\']=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024\n        except ImportError:state[\'peak_process_RSS_MiB\']=None\n        store.json(\'SCREEN_RESULTS.json\',state)\n    for family in sorted({f.split(\'__\')[1] for f,h in todo}):\n        columns=[c for c in CONFIG[\'candidate_features\'] if c.split(\'__\')[1]==family]\n        family_start=time.perf_counter()\n        xt,xdata,xhash=load_family(archives[\'X\'],store,family,columns,sources[\'X\'],checksums,CONFIG,io_workers)\n        state.setdefault(\'family_loading_seconds_this_session\',{})[family]=time.perf_counter()-family_start\n        for feature in [f for f in features if f.split(\'__\')[1]==family]:\n            xcol=xdata[:,columns.index(feature)]\n            for horizon in horizons:\n                if (feature,horizon) not in todo:continue\n                times,y,yhash=targets[horizon];indices=xt.get_indexer(times);assert (indices>=0).all();x=xcol[indices]\n                job_id=feature+\'|\'+str(horizon);job_name=\'job_\'+sha(job_id.encode())[:24]+\'.json\'\n                saved_job=fit_store.read(job_name)\n                job=json.loads(saved_job) if saved_job else {\'screen_signature\':sig,\'feature\':feature,\'horizon_minutes\':horizon,\'X_cache_sha256\':xhash,\'Y_cache_sha256\':yhash,\'fits\':{},\'folds\':{},\'matrix\':{}}\n                assert job[\'screen_signature\']==sig and job[\'feature\']==feature and job[\'horizon_minutes\']==horizon\n                assert job[\'X_cache_sha256\']==xhash and job[\'Y_cache_sha256\']==yhash\n                def save_job():fit_store.json(job_name,job)\n                for fold,(first,last) in enumerate(CONFIG[\'folds\']):\n                    key=f\'{feature}|{horizon}|{fold}\'\n                    if key in job[\'folds\']:continue\n                    masks=partition_masks(times,horizon,first,last);it,iv=masks[\'inner_train\'],masks[\'inner_valid\'];ot,ov=masks[\'outer_train\'],masks[\'outer_valid\']\n                    assert it.sum() and iv.sum() and ot.sum() and ov.sum()\n                    pending=[];inner=[]\n                    with threadpool_limits(limits=1),ThreadPoolExecutor(max_workers=fit_workers) as executor:\n                        for depth,leaf in CONFIG[\'grid\']:\n                            fit_key=f\'{key}|inner|{depth}|{leaf}\'\n                            if fit_key in job[\'fits\']:inner.append(job[\'fits\'][fit_key]);continue\n                            future=executor.submit(tree_fit,x[it],y[it],x[iv],y[iv],depth,leaf,CONFIG[\'minimum_observed_training\'])\n                            pending.append((future,fit_key))\n                        lookup={future:name for future,name in pending}\n                        for future in as_completed(lookup):\n                            score=future.result();job[\'fits\'][lookup[future]]=score;inner.append(score);save_job()\n                            print(f\'{horizon}m {feature} fold {fold+1}: inner setting saved\',flush=True)\n                    selected=pick_setting(inner);fit_key=key+\'|outer\'\n                    if fit_key not in job[\'fits\']:\n                        outer=tree_fit(x[ot],y[ot],x[ov],y[ov],selected[\'depth\'],selected[\'min_samples_leaf\'],CONFIG[\'minimum_observed_training\']) if selected else {\'status\':\'INNER_FEATURE_INELIGIBLE\',\'fit_count\':0}\n                        if outer[\'status\']!=\'PASS\':\n                            pred=float(np.median(y[ot]));errors=np.abs(y[ov]-pred)\n                            outer.update({\'fallback\':\'training_median_time_to_high\',\'MAE_minutes\':float(errors.mean()),\'absolute_error_sum\':float(errors.sum())})\n                        job[\'fits\'][fit_key]=outer;save_job()\n                    outer=job[\'fits\'][fit_key]\n                    job[\'folds\'][key]={\'feature\':feature,\'horizon_minutes\':horizon,\'fold\':fold+1,\n                        \'train_rows\':int(ot.sum()),\'validation_rows\':int(ov.sum()),\'inner_train_rows\':int(it.sum()),\'inner_validation_rows\':int(iv.sum()),\n                        \'inner_latest_training_label_end\':str((times[it]+pd.Timedelta(minutes=horizon)).max()),\n                        \'inner_latest_validation_label_end\':str((times[iv]+pd.Timedelta(minutes=horizon)).max()),\n                        \'outer_latest_training_label_end\':str((times[ot]+pd.Timedelta(minutes=horizon)).max()),\n                        \'selected\':None if selected is None else {\'depth\':selected[\'depth\'],\'min_samples_leaf\':selected[\'min_samples_leaf\']},\n                        \'MAE_minutes\':outer[\'MAE_minutes\'],\'absolute_error_sum\':outer[\'absolute_error_sum\'],\n                        \'status\':outer[\'status\'],\'X_cache_sha256\':xhash,\'Y_cache_sha256\':yhash,\'baseline_reproduction\':\'PASS\'}\n                    save_job();print(f\'{horizon}m {feature} fold {fold+1}: outer MAE={outer["MAE_minutes"]:.3f}\',flush=True)\n                update_matrix(job,baseline,dict(CONFIG,candidate_features=[feature],horizons=[horizon]))\n                save_job();state[\'matrix\'].update(job[\'matrix\'])\n                state[\'jobs\'][job_id]={\'fit_count\':sum(v.get(\'fit_count\',0) for v in job[\'fits\'].values()),\'fit_seconds\':sum(v.get(\'seconds\',0) for v in job[\'fits\'].values()),\'score_file\':job_name,\'score_file_id\':fit_store.ids[job_name],\n                                      \'score_sha256\':signature(job),\'status\':\'COMPLETE\'}\n                save()\n        del xdata\n    for k,v in CONFIG[\'source_ids\'].items():assert service.files().get(fileId=v,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()==sources[k]\n    state[\'status\']=\'PILOT_COMPLETE_REVIEW_BEFORE_FULL\' if mode==\'pilot\' else \'FULL_INDIVIDUAL_SCREEN_COMPLETE\';save()\n    print(\'\\nDONE: \'+state[\'status\'])\n    print(\'Horizon | feature | MAE minutes | delta vs lowest baseline (negative is better)\')\n    for feature in features:\n        for horizon in horizons:\n            row=state[\'matrix\'][f\'{feature}|{horizon}\'];print(horizon,feature,round(row[\'MAE_minutes\'],3),round(max(row[\'delta_MAE_vs_baselines\'].values()),3))\n    print(\'Send only this folder link: https://drive.google.com/drive/folders/\'+CONFIG[\'folder\'])\n', 'optimized': '"""Bounded execution and cache-only equivalence benchmark. No full-screen entry point.\n\nREF is the frozen screen_runner namespace, supplied by the notebook/test.\n"""\nimport json\nimport time\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nfrom collections import deque\nimport numpy as np\n\n\ndef scientific(score):\n    return {k: v for k, v in score.items() if k != \'seconds\'}\n\n\nclass OrderedWriter:\n    """One writer owns its store; serialize before queuing, bound pending snapshots."""\n    def __init__(self, write, capacity=2):\n        self.write = write\n        self.capacity = capacity\n        self.pool = ThreadPoolExecutor(max_workers=1)\n        self.pending = deque()\n        self.lock = threading.Lock()\n\n    def submit(self, name, value):\n        raw = json.dumps(value, allow_nan=False).encode()\n        with self.lock:\n            while len(self.pending) >= self.capacity:\n                self.pending.popleft().result()\n            self.pending.append(self.pool.submit(self.write, name, raw))\n\n    def close(self):\n        try:\n            while self.pending:\n                self.pending.popleft().result()\n        finally:\n            self.pool.shutdown(wait=True)\n\n\ndef prepare(x_train, y_train, x_valid, y_valid, minimum):\n    return (REF[\'preprocess\'](x_train, x_valid, minimum), y_train, y_valid,\n            np.isfinite(x_valid),\n            {\'train_observed\': int(np.isfinite(x_train).sum()), \'train_rows\': len(x_train),\n             \'validation_observed\': int(np.isfinite(x_valid).sum()), \'validation_rows\': len(x_valid)})\n\n\ndef prepared_fit(data, depth, leaf):\n    from sklearn.tree import DecisionTreeRegressor\n    start = time.perf_counter()\n    prepared, y_train, y_valid, observed, coverage = data\n    if prepared is None:\n        return {\'status\': \'INSUFFICIENT_TRAINING_COVERAGE_OR_VARIATION\',\n                \'fit_count\': 0, \'coverage\': coverage}, None\n    train, valid, median = prepared\n    model = DecisionTreeRegressor(criterion=\'absolute_error\', max_depth=depth,\n                                 min_samples_leaf=leaf, random_state=48)\n    model.fit(train, y_train)\n    prediction = model.predict(valid)\n    assert np.array_equal(prediction, REF[\'manual_predict\'](model, valid))\n    error = np.abs(prediction-y_valid)\n    score = {\'status\': \'PASS\', \'fit_count\': 1, \'depth\': depth, \'min_samples_leaf\': leaf,\n             \'MAE_minutes\': float(error.mean()), \'absolute_error_sum\': float(error.sum()),\n             \'observed_rows_MAE\': float(error[observed].mean()) if observed.any() else None,\n             \'missing_rows_MAE\': float(error[~observed].mean()) if (~observed).any() else None,\n             \'coverage\': coverage, \'imputation_median\': median,\n             \'tree_nodes\': int(model.tree_.node_count), \'seconds\': time.perf_counter()-start,\n             \'independent_tree_traversal\': \'PASS\'}\n    return score, prediction\n\n\ndef fold_task(x, y, masks, config, optimized, emit=lambda key, score: None):\n    """One fold per task; pool has at most two tasks, including outer fits.\n\n    Predictions are kept only in memory for equivalence checks, never checkpointed.\n    """\n    it, iv, ot, ov = [masks[k] for k in (\'inner_train\', \'inner_valid\', \'outer_train\', \'outer_valid\')]\n    minimum = config[\'minimum_observed_training\']\n    fits, predictions = {}, {}\n    data = prepare(x[it], y[it], x[iv], y[iv], minimum) if optimized else None\n    def fit(train, valid, depth, leaf, prepared):\n        if optimized:\n            return prepared_fit(prepared, depth, leaf)\n        # Capture the frozen implementation\'s independent prediction traversal.\n        local = {}\n        original = REF[\'manual_predict\']\n        def capture(model, values):\n            prediction = original(model, values)\n            local[\'prediction\'] = prediction\n            return prediction\n        import types\n        namespace = dict(REF[\'tree_fit\'].__globals__, manual_predict=capture)\n        reference = types.FunctionType(REF[\'tree_fit\'].__code__, namespace)\n        score = reference(x[train], y[train], x[valid], y[valid], depth, leaf, minimum)\n        return score, local.get(\'prediction\')\n    for depth, leaf in config[\'grid\']:\n        key = f\'inner|{depth}|{leaf}\'\n        fits[key], predictions[key] = fit(it, iv, depth, leaf, data)\n        emit(key, fits[key])\n    selected = REF[\'pick_setting\'](list(fits.values()))\n    data = prepare(x[ot], y[ot], x[ov], y[ov], minimum) if optimized and selected else None\n    outer, prediction = fit(ot, ov, selected[\'depth\'], selected[\'min_samples_leaf\'], data) if selected else ({\'status\': \'INNER_FEATURE_INELIGIBLE\', \'fit_count\': 0}, None)\n    if outer[\'status\'] != \'PASS\':\n        prediction = np.full(int(ov.sum()), float(np.median(y[ot])))\n        error = np.abs(y[ov]-prediction)\n        outer.update(fallback=\'training_median_time_to_high\', MAE_minutes=float(error.mean()), absolute_error_sum=float(error.sum()))\n    fits[\'outer\'], predictions[\'outer\'] = outer, prediction\n    emit(\'outer\', outer)\n    return {\'fits\': fits, \'predictions\': predictions,\n            \'selected\': None if selected is None else [selected[\'depth\'], selected[\'min_samples_leaf\']]}\n\n\ndef assert_equal(a, b):\n    assert a[\'selected\'] == b[\'selected\']\n    assert a[\'fits\'].keys() == b[\'fits\'].keys()\n    for key in a[\'fits\']:\n        assert scientific(a[\'fits\'][key]) == scientific(b[\'fits\'][key]), key\n        left, right = a[\'predictions\'][key], b[\'predictions\'][key]\n        assert (left is None and right is None) or np.array_equal(left, right), key\n\n\ndef run_benchmark(config, reference_hash, optimized_hash, expected_pilot_identity):\n    import os\n    import platform\n    import pandas as pd\n    import sklearn\n    import pyarrow\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from threadpoolctl import threadpool_limits\n    auth.authenticate_user()\n    credentials, _ = google.auth.default()\n    service = build(\'drive\', \'v3\', credentials=credentials, cache_discovery=False)\n    store = REF[\'DriveStore\'](service, config[\'folder\'])\n    saved = json.loads(store.read(\'SCREEN_RESULTS.json\'))\n    versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,\n                    sklearn=sklearn.__version__, pyarrow=pyarrow.__version__)\n    assert saved[\'identity\'] == expected_pilot_identity, \'Pilot identity changed\'\n    assert saved[\'identity\'][\'config\'] == config and saved[\'identity\'][\'sources\'] == config[\'audit_sources\']\n    assert saved[\'identity\'][\'versions\'] == versions, \'Runtime changed; stop for review. Do not replace pilot.\'\n    assert REF[\'signature\'](saved[\'identity\']) == saved[\'signature\']\n    prior = store.read(\'THROUGHPUT_CHECK.json\')\n    if prior:\n        receipt = json.loads(prior)\n        if (receipt.get(\'status\') == \'PASS_EXACT_PREDICTIONS_SCORES_SELECTION_AND_SAVED_PILOT\'\n                and receipt.get(\'reference_sha256\') == reference_hash\n                and receipt.get(\'optimized_sha256\') == optimized_hash\n                and receipt.get(\'screen_signature\') == saved[\'signature\']):\n            print(\'This verification already passed. Reusing receipt; no fits repeated.\')\n            return\n    for key, file_id in config[\'source_ids\'].items():\n        current = service.files().get(fileId=file_id, fields=\'id,name,size,md5Checksum,modifiedTime\').execute()\n        assert current == config[\'audit_sources\'][key]\n    baseline = json.loads(store.read(\'BASELINE_RESULTS.json\'))\n    assert REF[\'signature\'](baseline[\'identity\']) == baseline[\'signature\'] == config[\'baseline_signature\']\n    columns = [f for f in config[\'candidate_features\'] if f.split(\'__\')[1] == \'flow\']\n    identity = dict(type=\'f4z_research_cache_v1\', source=config[\'audit_sources\'][\'X\'], columns=columns,\n                    origin_end=config[\'origin_end_exclusive\'], cadence=30, minimum_baseline_n=1000,\n                    allowed_source_status=[\'PROVISIONAL\'])\n    meta, xt, matrix = REF[\'read_cache\'](store, \'SCREEN_X_F4Z_flow.npz\', identity)\n    assert meta[\'completed\'] == config[\'months\'] and xt is not None\n    xt = pd.DatetimeIndex(xt, tz=\'UTC\')\n    assert xt.is_unique and xt.is_monotonic_increasing and (xt < pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n    tasks, existing = [], []\n    # Fixed before seeing scores: each pilot feature paired with one pilot horizon,\n    # latest outer fold (largest training population); 21 fits per implementation.\n    for feature, horizon in zip(config[\'pilot_features\'], config[\'pilot_horizons\']):\n        identity = dict(type=\'valid_time_high_research_v1\', source=config[\'audit_sources\'][\'Y\'],\n                        horizon=horizon, origin_end=config[\'origin_end_exclusive\'], cadence=30)\n        ym, times, y = REF[\'read_cache\'](store, f\'SCREEN_Y_{horizon}m.npz\', identity)\n        assert ym[\'completed\'] == config[\'months\'] and times is not None\n        times = pd.DatetimeIndex(times, tz=\'UTC\')\n        assert times.is_unique and times.is_monotonic_increasing\n        assert (times < pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n        assert ((times+pd.Timedelta(minutes=horizon)) < pd.Timestamp(config[\'reserve_start\'])).all()\n        REF[\'check_baselines\'](times, y, horizon, baseline, config)\n        indices = xt.get_indexer(times)\n        assert (indices >= 0).all()\n        masks = REF[\'partition_masks\'](times, horizon, *config[\'folds\'][-1])\n        tasks.append((matrix[indices, columns.index(feature)], y, masks, config))\n        job_id = feature+\'|\'+str(horizon)\n        info = saved[\'jobs\'][job_id]\n        job = json.loads(service.files().get_media(fileId=info[\'score_file_id\']).execute())\n        assert REF[\'signature\'](job) == info[\'score_sha256\']\n        assert job[\'screen_signature\'] == saved[\'signature\']\n        assert job[\'X_cache_sha256\'] == meta[\'array_sha256\'] and job[\'Y_cache_sha256\'] == ym[\'array_sha256\']\n        existing.append((job_id, job))\n    output = {\'status\': \'RUNNING_EQUIVALENCE_ONLY\', \'reference_sha256\': reference_hash,\n              \'optimized_sha256\': optimized_hash, \'screen_signature\': saved[\'signature\'],\n              \'pilot_declared_runner_sha256\': saved[\'identity\'][\'runner_sha256\'],\n              \'reference_hash_matches_pilot_declaration\': reference_hash == saved[\'identity\'][\'runner_sha256\'],\n              \'versions\': versions, \'scope\': \'three fixed pilot pairs, outer fold 5, 42 total fits\',\n              \'full_run_authorized\': False, \'scores\': {}, \'timings\': {}}\n    output[\'fit_workers\'] = min(2, os.cpu_count() or 1)\n    # Dedicated service is owned exclusively by the single writer thread.\n    writer_store = REF[\'DriveStore\'](build(\'drive\', \'v3\', credentials=credentials, cache_discovery=False), config[\'folder\'])\n    writer = OrderedWriter(writer_store.write)\n    lock = threading.Lock()\n    def emit(task_id, key, score):\n        with lock:\n            output[\'scores\'][f\'{task_id}|{key}\'] = score\n            writer.submit(\'THROUGHPUT_CHECK.json\', output)\n    results = {}\n    try:\n        with threadpool_limits(limits=1):\n            for optimized in (False, True):\n                label = \'optimized\' if optimized else \'reference\'\n                start = time.perf_counter()\n                with ThreadPoolExecutor(max_workers=output[\'fit_workers\']) as pool:\n                    futures = [pool.submit(fold_task, *task, optimized,\n                               lambda key, score, i=i: emit(f\'{label}|{i}\', key, score))\n                               for i, task in enumerate(tasks)]\n                    results[label] = [f.result() for f in futures]\n                output[\'timings\'][label+\'_compute_and_enqueue_seconds\'] = time.perf_counter()-start\n                # Barrier: include all remote checkpoint completion in each timing.\n                writer.close()\n                output[\'timings\'][label+\'_including_upload_seconds\'] = time.perf_counter()-start\n                writer = OrderedWriter(writer_store.write)\n                print(label+\' phase complete\', flush=True)\n        for i, (a, b) in enumerate(zip(results[\'reference\'], results[\'optimized\'])):\n            assert_equal(a, b)\n            job_id, job = existing[i]\n            for key, score in a[\'fits\'].items():\n                assert scientific(score) == scientific(job[\'fits\'][f\'{job_id}|4|{key}\'])\n        output[\'status\'] = \'PASS_EXACT_PREDICTIONS_SCORES_SELECTION_AND_SAVED_PILOT\'\n        output[\'limitations\'] = [\'Three fixed cases only; not full-universe equivalence or a runtime guarantee.\',\n                                \'Both phases use bounded fold scheduling and asynchronous writes; timing isolates preparation reuse, not total speedup over the old scheduler.\',\n                                \'No automatic checkpoint migration or full launch. Review this receipt first.\',\n                                \'Current reference content hash differs from historical pilot declaration; saved-score replay is required, not identity equivalence.\',\n                                \'An interrupted verification may repeat at most 42 fits; a matching PASS receipt is reused.\']\n        writer.submit(\'THROUGHPUT_CHECK.json\', output)\n    except BaseException:\n        output[\'status\'] = \'FAILED_OR_INTERRUPTED_NOT_APPROVED\'\n        # Drain earlier snapshots before attempting a final failure receipt.\n        # A failed upload must not be hidden by a secondary shutdown error.\n        try:\n            writer.close()\n        except Exception:\n            pass\n        writer = OrderedWriter(writer_store.write)\n        try:\n            writer.submit(\'THROUGHPUT_CHECK.json\', output)\n            writer.close()\n        except Exception:\n            print(\'Failure receipt could not be uploaded; this run is NOT verified.\')\n        raise\n    finally:\n        writer.close()\n    print(output[\'status\'])\n    print(json.dumps(output[\'timings\'], indent=2))\n    print(\'Saved THROUGHPUT_CHECK.json. Full screen remains disabled.\')\n', 'report': '"""Static scientific report and complete CSV tables; no website or raw data."""\nimport csv\nimport io\nimport json\nimport numpy as np\n\n\ndef tables(config, baseline, jobs, horizon=15):\n    base = baseline[\'completed\'][str(horizon)]\n    policy = min(base[\'pooled\'], key=lambda k:base[\'pooled\'][k][\'MAE_minutes\'])\n    best = base[\'pooled\'][policy][\'MAE_minutes\']\n    rows, folds = [], []\n    for feature in config[\'candidate_features\']:\n        job = jobs.get(feature+\'|\'+str(horizon), {})\n        entries = [job.get(\'folds\',{}).get(f\'{feature}|{horizon}|{i}\') for i in range(len(config[\'folds\']))]\n        row = {\'rank\':\'\',\'feature\':feature,\'family\':feature.split(\'__\')[1],\n               \'feature_lookback\':feature.split(\'__\')[3:-1], \'target\':config[\'target\'],\n               \'forecast_horizon_minutes\':horizon,\'status\':\'PENDING\',\n               \'completed_folds\':sum(e is not None for e in entries),\n               \'MAE_minutes\':\'\',\'MAE_percent_of_horizon\':\'\',\'improvement_minutes\':\'\',\'improvement_percent\':\'\',\n               \'folds_beating_reference_policy\':\'\',\'observed_validation_percent\':\'\',\'fallback_folds\':\'\',\n               \'baseline_policy_for_comparison\':policy,\'baseline_MAE_minutes\':best,\n               \'model\':\'DecisionTreeRegressor absolute_error seed48\',\n               \'transformation\':\'Raw feature + missing indicator; train-only median imputation\',\n               \'selected_settings_by_fold\':\'\',\'verification\':\'Pending\',\n               \'limitations\':\'Provisional development; upstream lineage unresolved; ranking is not significance or trading evidence\'}\n        for name,v in base[\'pooled\'].items(): row[\'baseline_\'+name+\'_MAE_minutes\']=v[\'MAE_minutes\']\n        if row[\'completed_folds\']: row[\'status\']=\'PARTIAL\'\n        for i,e in enumerate(entries):\n            if e is None: continue\n            score = job[\'fits\'][f\'{feature}|{horizon}|{i}|outer\']\n            coverage = score.get(\'coverage\',{})\n            folds.append({\'feature\':feature,\'horizon_minutes\':horizon,\'fold\':i+1,\n                          \'period_start\':config[\'folds\'][i][0], \'period_end_exclusive\':config[\'folds\'][i][1],\n                          \'MAE_minutes\':e[\'MAE_minutes\'],\'validation_rows\':e[\'validation_rows\'],\n                          \'baseline_policy\':policy,\'baseline_MAE_minutes\':base[\'folds\'][i][\'scores\'][policy][\'MAE_minutes\'],\n                          \'selected_settings\':e[\'selected\'], \'status\':e[\'status\'],\n                          \'observed_validation_rows\':coverage.get(\'validation_observed\',\'\'),\n                          \'observed_training_rows\':coverage.get(\'train_observed\',\'\'),\n                          \'latest_training_label_end\':e[\'outer_latest_training_label_end\']})\n        if all(e is not None for e in entries):\n            count=sum(e[\'validation_rows\'] for e in entries)\n            mae=sum(e[\'absolute_error_sum\'] for e in entries)/count\n            assert count==next(iter(base[\'pooled\'].values()))[\'rows\']\n            cover=[job[\'fits\'][f\'{feature}|{horizon}|{i}|outer\'].get(\'coverage\',{}).get(\'validation_observed\') for i in range(len(entries))]\n            row.update(status=\'COMPLETE\',MAE_minutes=mae,MAE_percent_of_horizon=100*mae/horizon,\n                       improvement_minutes=best-mae,improvement_percent=100*(best-mae)/best,\n                       folds_beating_reference_policy=sum(e[\'MAE_minutes\']<base[\'folds\'][i][\'scores\'][policy][\'MAE_minutes\'] for i,e in enumerate(entries)),\n                       observed_validation_percent=100*sum(cover)/count if all(v is not None for v in cover) else \'\',\n                       fallback_folds=sum(e[\'status\']!=\'PASS\' for e in entries),\n                       selected_settings_by_fold=[e[\'selected\'] for e in entries],\n                       verification=\'Receipt checks; tree traversal reported for fitted models; no full raw-data replay\')\n        rows.append(row)\n    ranked=sorted((r for r in rows if r[\'status\']==\'COMPLETE\'),key=lambda r:(r[\'MAE_minutes\'],r[\'feature\']))\n    for i,row in enumerate(ranked,1): row[\'rank\']=i\n    rows=ranked+[r for r in rows if r[\'status\']!=\'COMPLETE\']\n    return rows,folds,policy\n\n\ndef csv_bytes(rows):\n    buffer=io.StringIO(newline=\'\')\n    if rows:\n        writer=csv.DictWriter(buffer,fieldnames=list(rows[0])); writer.writeheader()\n        for row in rows:\n            writer.writerow({k:json.dumps(v) if isinstance(v,(list,dict)) else v for k,v in row.items()})\n    return buffer.getvalue().encode(\'utf-8-sig\')\n\n\ndef artifacts(config,baseline,jobs,horizon=15):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from matplotlib.backends.backend_pdf import PdfPages\n    rows,folds,policy=tables(config,baseline,jobs,horizon)\n    ranked=[r for r in rows if r[\'status\']==\'COMPLETE\']\n    buf=io.BytesIO()\n    with PdfPages(buf) as pdf:\n        fig=plt.figure(figsize=(11.7,8.3)); fig.patch.set_facecolor(\'#f5f7fa\')\n        fig.text(.07,.91,\'BTCUSDT | Timing research\',fontsize=25,weight=\'bold\',color=\'#15304c\')\n        fig.text(.07,.85,f\'Target: minutes until the earliest high | Current forecast: {horizon} minutes\',fontsize=13)\n        fig.text(.07,.77,f\'{len(ranked)} / {len(rows)} features complete\',fontsize=23,weight=\'bold\')\n        fig.text(.07,.72,f\'{sum(r["status"]=="PARTIAL" for r in rows)} partial | {sum(r["status"]=="PENDING" for r in rows)} pending\',fontsize=13)\n        summary=[]\n        for h in config[\'horizons\']:\n            count=sum(len(j.get(\'folds\',{}))==len(config[\'folds\']) for j in jobs.values() if j[\'horizon_minutes\']==h)\n            summary.append([\'15m\' if h==15 else f\'{h//60}h\',f\'{count} / {len(rows)}\', \'Current screen\' if h==horizon else (\'Pilot only\' if count else \'Pending\')])\n        ax=fig.add_axes([.07,.24,.86,.41]); ax.axis(\'off\')\n        table=ax.table(cellText=summary,colLabels=[\'Forecast horizon\',\'Completed features\',\'Stage\'],loc=\'center\',cellLoc=\'left\'); table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1,1.6)\n        fig.text(.07,.14,\'MAE = average absolute timing error; lower is better. It is not the predicted time.\',fontsize=11)\n        fig.text(.07,.10,\'Feature lookback describes past data; forecast horizon describes the future window.\',fontsize=11)\n        fig.text(.07,.06,\'Development results only. Reserved six months excluded. No significance or profitability claim.\',fontsize=10,color=\'#6b4d00\')\n        pdf.savefig(fig); plt.close(fig)\n        fig,axes=plt.subplots(2,1,figsize=(11.7,10),gridspec_kw={\'height_ratios\':[1.2,1]})\n        fig.suptitle(f\'{horizon}-minute forecast | completed features only\',fontsize=19,weight=\'bold\')\n        top=ranked[:15]\n        if top:\n            labels=[r[\'feature\'].replace(\'F4Z__\',\'\').replace(\'__value\',\'\').replace(\'__\',\' / \') for r in top]\n            values=[r[\'improvement_minutes\'] for r in top]\n            axes[0].barh(labels,values,color=[\'#127c70\' if v>0 else \'#bd5b32\' for v in values]); axes[0].invert_yaxis(); axes[0].axvline(0,color=\'#172c45\',lw=1)\n            axes[0].set_xlabel(\'Minutes of error reduced versus best pooled baseline (positive = better)\')\n            axes[0].set_title(\'Top ranked features — a top rank may still lose to the baseline\',fontsize=11)\n            matrix=np.array([[baseline[\'completed\'][str(horizon)][\'folds\'][i][\'scores\'][policy][\'MAE_minutes\']-jobs[r[\'feature\']+\'|\'+str(horizon)][\'folds\'][f\'{r["feature"]}|{horizon}|{i}\'][\'MAE_minutes\'] for i in range(5)] for r in top])\n            limit=max(abs(matrix).max(),.001)\n            plot=axes[1].imshow(matrix,cmap=\'BrBG\',vmin=-limit,vmax=limit,aspect=\'auto\')\n            axes[1].set_yticks(range(len(top)),labels=labels,fontsize=8)\n            axes[1].set_xticks(range(5),labels=[v[0][:7] for v in config[\'folds\']])\n            axes[1].set_title(\'Consistency: improvement against the same baseline policy in each period\',fontsize=11)\n            for i in range(len(top)):\n                for j in range(5): axes[1].text(j,i,f\'{matrix[i,j]:+.3f}\',ha=\'center\',va=\'center\',fontsize=7)\n            fig.colorbar(plot,ax=axes[1],label=\'Minutes of error reduced\')\n        else:\n            for ax in axes: ax.text(.5,.5,\'No completed feature results yet\',ha=\'center\',transform=ax.transAxes); ax.axis(\'off\')\n        fig.tight_layout(rect=[.02,.02,.98,.94]); pdf.savefig(fig); plt.close(fig)\n        fig=plt.figure(figsize=(11.7,8.3)); fig.text(.06,.93,\'Coverage, interpretation and verification\',fontsize=21,weight=\'bold\')\n        lines=[f\'Comparison policy: {policy}\',\n               \'The reference policy is chosen by pooled development MAE, not a deployable future selector.\',\n               \'Green / positive means less error. Negative means the baseline performed better.\',\n               \'Rankings include complete five-fold results only. Partial and pending features are in the CSV.\',\n               \'All three baseline MAEs, fold settings and coverage are provided in the complete tables.\',\n               \'Fallback rows remain in evaluation. Unknown legacy fallback coverage is left blank.\',\n               \'Earlier-only training and inner selection; strict label-end purging; no reserved rows.\',\n               \'Source availability gates are applied, but original F4Z publication/normalization lineage\',\n               \'remains unresolved. Features and results are PROVISIONAL.\',\n               \'Selecting among many features on development data can overfit. No fresh-test claim.\',\n               \'A standalone feature screen does not test feature sets or connected prediction models.\']\n        for i,line in enumerate(lines): fig.text(.06,.85-i*.045,line,fontsize=11)\n        ax=fig.add_axes([.06,.06,.88,.25]); ax.axis(\'off\')\n        view=[[r[\'rank\'],r[\'feature\'].replace(\'F4Z__\',\'\').replace(\'__value\',\'\'),f\'{r["MAE_minutes"]:.4f}\',f\'{r["folds_beating_reference_policy"]}/5\',f\'{r["observed_validation_percent"]:.1f}%\' if r[\'observed_validation_percent\']!=\'\' else \'Unknown\',r[\'fallback_folds\']] for r in ranked[:5]]\n        if view:\n            tab=ax.table(cellText=view,colLabels=[\'Rank\',\'Feature\',\'MAE min\',\'Wins\',\'Observed\',\'Fallback folds\'],loc=\'center\',colWidths=[.05,.48,.12,.08,.12,.15]); tab.auto_set_font_size(False); tab.set_fontsize(8); tab.scale(1,1.8)\n        pdf.savefig(fig); plt.close(fig)\n    return {f\'{horizon}m_summary.pdf\':buf.getvalue(),f\'{horizon}m_feature_rankings.csv\':csv_bytes(rows),\n            f\'{horizon}m_fold_details.csv\':csv_bytes(folds)}\n', 'runner': '"""1h-only resumable screen. REF/OPT/REPORT namespaces supplied by notebook."""\nimport json\nimport time\nimport threading\nimport platform\nfrom concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED\nimport numpy as np\nimport pandas as pd\n\n\nclass BudgetStop(Exception):\n    pass\n\n\ndef pack(job):\n    return {\'job\':job,\'sha256\':REF[\'signature\'](job)}\n\n\ndef unpack(raw, signature, feature, xhash, yhash):\n    record=json.loads(raw); job=record[\'job\']\n    assert REF[\'signature\'](job)==record[\'sha256\'], \'Job checksum mismatch\'\n    assert job[\'screen_signature\']==signature and job[\'feature\']==feature and job[\'horizon_minutes\']==60\n    assert job[\'X_cache_sha256\']==xhash and job[\'Y_cache_sha256\']==yhash\n    return job\n\n\ndef recover_indexed(record, info, signature, key):\n    job=record[\'job\']\n    assert REF[\'signature\'](job)==record[\'sha256\']\n    if info[\'complete_folds\']==5: assert record[\'sha256\']==info[\'sha256\']\n    assert job[\'screen_signature\']==signature and job[\'feature\']+\'|60\'==key\n    assert len(job[\'folds\'])>=info[\'complete_folds\'] and job[\'horizon_minutes\']==60\n    info.update(sha256=record[\'sha256\'],complete_folds=len(job[\'folds\']))\n    return job\n\n\ndef fit_job(job, x, times, y, masks, config, checkpoint, deadline, clock=time.monotonic):\n    """One worker owns one job. Saved scores skip fits; incomplete aggregates recover."""\n    feature=job[\'feature\']; minimum=config[\'minimum_observed_training\']\n    for i,m in enumerate(masks):\n        prefix=f\'{feature}|60|{i}\'\n        if prefix in job[\'folds\']: continue\n        it,iv,ot,ov=[m[k] for k in (\'inner_train\',\'inner_valid\',\'outer_train\',\'outer_valid\')]\n        assert it.any() and iv.any() and ot.any() and ov.any()\n        inner=[]; prepared=None\n        for depth,leaf in config[\'grid\']:\n            key=f\'{prefix}|inner|{depth}|{leaf}\'\n            if key not in job[\'fits\']:\n                if clock()>=deadline: return job\n                if prepared is None: prepared=OPT[\'prepare\'](x[it],y[it],x[iv],y[iv],minimum)\n                score,prediction=OPT[\'prepared_fit\'](prepared,depth,leaf); del prediction\n                job[\'fits\'][key]=score; checkpoint(job)\n                print(f\'{feature} fold {i+1}: inner {depth}/{leaf} complete\',flush=True)\n            inner.append(job[\'fits\'][key])\n        selected=REF[\'pick_setting\'](inner); key=prefix+\'|outer\'\n        if key not in job[\'fits\']:\n            if clock()>=deadline: return job\n            coverage={\'train_observed\':int(np.isfinite(x[ot]).sum()),\'train_rows\':int(ot.sum()),\n                      \'validation_observed\':int(np.isfinite(x[ov]).sum()),\'validation_rows\':int(ov.sum())}\n            if selected:\n                score,prediction=OPT[\'prepared_fit\'](OPT[\'prepare\'](x[ot],y[ot],x[ov],y[ov],minimum),selected[\'depth\'],selected[\'min_samples_leaf\']); del prediction\n            else: score={\'status\':\'INNER_FEATURE_INELIGIBLE\',\'fit_count\':0,\'coverage\':coverage}\n            if score[\'status\']!=\'PASS\':\n                errors=np.abs(y[ov]-float(np.median(y[ot])))\n                score.update(fallback=\'training_median_time_to_high\',MAE_minutes=float(errors.mean()),absolute_error_sum=float(errors.sum()))\n            job[\'fits\'][key]=score; checkpoint(job)\n        outer=job[\'fits\'][key]\n        job[\'folds\'][prefix]={\'feature\':feature,\'horizon_minutes\':60,\'fold\':i+1,\n            \'train_rows\':int(ot.sum()),\'validation_rows\':int(ov.sum()),\'inner_train_rows\':int(it.sum()),\'inner_validation_rows\':int(iv.sum()),\n            \'inner_latest_training_label_end\':str((times[it]+pd.Timedelta(minutes=60)).max()),\n            \'inner_latest_validation_label_end\':str((times[iv]+pd.Timedelta(minutes=60)).max()),\n            \'outer_latest_training_label_end\':str((times[ot]+pd.Timedelta(minutes=60)).max()),\n            \'selected\':None if selected is None else {\'depth\':selected[\'depth\'],\'min_samples_leaf\':selected[\'min_samples_leaf\']},\n            \'MAE_minutes\':outer[\'MAE_minutes\'],\'absolute_error_sum\':outer[\'absolute_error_sum\'],\n            \'status\':outer[\'status\'],\'X_cache_sha256\':job[\'X_cache_sha256\'],\'Y_cache_sha256\':job[\'Y_cache_sha256\'],\n            \'baseline_reproduction\':\'PASS\'}\n        checkpoint(job)\n        print(f\'{feature} fold {i+1}/5 complete\',flush=True)\n    return job\n\n\ndef folder(service,parent,name):\n    found=service.files().list(q=f"\'{parent}\' in parents and name=\'{name}\' and mimeType=\'application/vnd.google-apps.folder\' and trashed=false",fields=\'files(id)\').execute()[\'files\']\n    assert len(found)<=1\n    return found[0][\'id\'] if found else service.files().create(body={\'name\':name,\'parents\':[parent],\'mimeType\':\'application/vnd.google-apps.folder\'},fields=\'id\').execute()[\'id\']\n\n\nclass IsolatedStore:\n    """Write only to this horizon; borrow only complete shared caches read-only."""\n    def __init__(self, private, shared, months):\n        self.private,self.shared,self.months=private,shared,months\n    def read(self,name):\n        raw=self.private.read(name)\n        if raw is not None: return raw\n        if not (name.startswith(\'SCREEN_X_F4Z_\') or name==\'SCREEN_Y_60m.npz\'):\n            return None\n        raw=self.shared.read(name)\n        if raw is None: return None\n        import io\n        with np.load(io.BytesIO(raw),allow_pickle=False) as cache:\n            meta=json.loads(str(cache[\'metadata\'].item()))\n        if meta[\'completed\']!=self.months: return None\n        # REF.read_cache verifies source identity and full array checksum before use.\n        return raw\n    def write(self,name,data,mime=\'application/json\'):\n        return self.private.write(name,data,mime)\n    def json(self,name,value):\n        return self.private.json(name,value)\n\n\ndef run_1h(config, expected_pilot, hashes, session_hours=4):\n    import sklearn\n    import pyarrow\n    import joblib\n    from sklearn.tree import DecisionTreeRegressor\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from google_auth_httplib2 import AuthorizedHttp\n    import httplib2\n    from threadpoolctl import threadpool_limits\n    assert config[\'target\']==\'time_to_high_min\' and 60 in config[\'horizons\']\n    assert 0<session_hours<=4\n    start=time.monotonic(); deadline=start+session_hours*3600\n    auth.authenticate_user(); credentials,_=google.auth.default()\n    def service_new():\n        return build(\'drive\',\'v3\',http=AuthorizedHttp(credentials,http=httplib2.Http(timeout=90)),cache_discovery=False)\n    service=service_new(); store=REF[\'DriveStore\'](service,config[\'folder\'])\n    print(\'1h ONLY: checking identities, pilot receipts and verified implementation.\',flush=True)\n    pilot=json.loads(store.read(\'SCREEN_RESULTS.json\'))\n    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__,pyarrow=pyarrow.__version__)\n    assert pilot[\'identity\']==expected_pilot and REF[\'signature\'](expected_pilot)==pilot[\'signature\']\n    assert expected_pilot[\'config\']==config and expected_pilot[\'versions\']==versions\n    for k,file_id in config[\'source_ids\'].items():\n        assert service.files().get(fileId=file_id,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()==config[\'audit_sources\'][k]\n    receipt=json.loads(store.read(\'THROUGHPUT_CHECK.json\'))\n    assert receipt[\'status\']==\'PASS_EXACT_PREDICTIONS_SCORES_SELECTION_AND_SAVED_PILOT\'\n    assert receipt[\'screen_signature\']==pilot[\'signature\'] and receipt[\'reference_sha256\']==hashes[\'reference\'] and receipt[\'optimized_sha256\']==hashes[\'optimized\']\n    workers_receipt=json.loads(store.read(\'WORKER_CHECK.json\'))\n    assert workers_receipt[\'status\']==\'PASS_WORKER_EQUIVALENCE\'\n    assert REF[\'signature\'](workers_receipt[\'identity\'])==workers_receipt[\'signature\']\n    assert workers_receipt[\'identity\'][\'pilot_signature\']==pilot[\'signature\']\n    assert workers_receipt[\'identity\'][\'hashes\'][\'reference\']==hashes[\'reference\'] and workers_receipt[\'identity\'][\'hashes\'][\'optimized\']==hashes[\'optimized\']\n    baseline=json.loads(store.read(\'BASELINE_RESULTS.json\'))\n    assert REF[\'signature\'](baseline[\'identity\'])==baseline[\'signature\']==config[\'baseline_signature\']\n    assert baseline[\'identity\'][\'source\']==config[\'audit_sources\'][\'Y\']\n    shared_store=store\n    work_root=folder(service,config[\'folder\'],\'Timing_1h_Run\')\n    store=IsolatedStore(REF[\'DriveStore\'](service,work_root),shared_store,config[\'months\'])\n    identity={\'target\':config[\'target\'],\'horizon_minutes\':60,\'pilot_signature\':pilot[\'signature\'],\n              \'implementation_hashes\':{k:hashes[k] for k in (\'reference\',\'optimized\',\'runner\',\'range_reader\')},\'versions\':versions,\n              \'compatibility\':\'Reuse only individually hash-verified legacy jobs under their original identity; new jobs use this identity. Historical runner hash discrepancy retained, never rewritten.\'}\n    sig=REF[\'signature\'](identity)\n    raw=store.read(\'SCREEN_1h_RESULTS.json\')\n    state=json.loads(raw) if raw else {\'identity\':identity,\'signature\':sig,\'jobs\':{},\'status\':\'RUNNING_1h_ONLY\'}\n    assert state[\'identity\']==identity and state[\'signature\']==sig, \'1h implementation changed; review compatibility before reuse\'\n    scores_folder=folder(service,work_root,\'Timing_1h_Fit_Scores\')\n    reports_folder=folder(service,work_root,\'Timing_Research_Reports\')\n    read_scores=REF[\'DriveStore\'](service,scores_folder)\n    report_store=REF[\'DriveStore\'](service,reports_folder)\n    # One HTTP client owned by one asynchronous writer. Main reads use another.\n    write_scores=REF[\'DriveStore\'](service_new(),scores_folder)\n    all_jobs={}\n    for key,info in pilot[\'jobs\'].items():\n        job=json.loads(service.files().get_media(fileId=info[\'score_file_id\']).execute())\n        assert REF[\'signature\'](job)==info[\'score_sha256\'] and job[\'screen_signature\']==pilot[\'signature\']\n        all_jobs[key]=job\n    def save_reports():\n        for name,data in REPORT[\'artifacts\'](config,baseline,all_jobs,60).items():\n            report_store.write(name,data,\'application/pdf\' if name.endswith(\'.pdf\') else \'text/csv\')\n        print(\'Reports updated: \'+f\'https://drive.google.com/drive/folders/{reports_folder}\',flush=True)\n    # Validate completed new jobs before skipping. Load partials later using cache hashes.\n    for key,info in state[\'jobs\'].items():\n        raw=service.files().get_media(fileId=info[\'file_id\']).execute(); record=json.loads(raw)\n        # A resumed partial job can advance before its aggregate is saved.\n        # Completed jobs are immutable; partial aggregates may lag durable fits.\n        job=recover_indexed(record,info,sig,key)\n        all_jobs[key]=job\n    pending=[f for f in config[\'candidate_features\'] if len(all_jobs.get(f+\'|60\',{}).get(\'folds\',{}))!=5]\n    if not pending:\n        state[\'status\']=\'COMPLETE_1h_ONLY\'; store.json(\'SCREEN_1h_RESULTS.json\',state); save_reports(); return\n    yidentity=dict(type=\'valid_time_high_research_v1\',source=config[\'audit_sources\'][\'Y\'],horizon=60,origin_end=config[\'origin_end_exclusive\'],cadence=30)\n    ym,ts,y=REF[\'read_cache\'](store,\'SCREEN_Y_60m.npz\',yidentity)\n    if ym[\'completed\']!=config[\'months\']:\n        class TargetBudgetStore:\n            def read(self,name): return store.read(name)\n            def write(self,name,data,mime=\'application/json\'):\n                store.write(name,data,mime)\n                if time.monotonic()>=deadline: raise BudgetStop(\'Target month saved; resume next session\')\n        try:\n            if time.monotonic()>=deadline: raise BudgetStop(\'Session budget reached\')\n            print(\'Loading 1h target via remote ZIP ranges; research months only.\',flush=True)\n            yarchive=REF[\'RemoteArchive\'](credentials,config[\'audit_sources\'][\'Y\'])\n            manifest_name=next(n for n in yarchive.members if n.endswith(\'/HANDOFF_2_MANIFEST.json\'))\n            yroot=manifest_name[:-len(\'HANDOFF_2_MANIFEST.json\')]\n            manifest={yroot+entry[\'path\']:entry[\'sha256\'] for entry in json.loads(yarchive.read(manifest_name))[\'files\']}\n            loaded_times,y,yhash=REF[\'load_target\'](yarchive,TargetBudgetStore(),yroot,manifest,60,config[\'audit_sources\'][\'Y\'],config)\n            ts=loaded_times.asi8\n            ym,_,_=REF[\'read_cache\'](store,\'SCREEN_Y_60m.npz\',yidentity)\n        except BudgetStop:\n            state[\'status\']=\'PAUSED_TARGET_LOADING_1h_ONLY\'\n            store.json(\'SCREEN_1h_RESULTS.json\',state); save_reports()\n            print(\'Target-loading progress saved. Rerun the same 1h notebook to resume.\',flush=True)\n            return\n    assert ym[\'completed\']==config[\'months\'] and ts is not None\n\n    times=pd.DatetimeIndex(ts,tz=\'UTC\'); yhash=ym[\'array_sha256\']\n    assert times.is_unique and times.is_monotonic_increasing and (times<pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n    assert ((times+pd.Timedelta(minutes=60))<pd.Timestamp(config[\'reserve_start\'])).all()\n    assert np.isfinite(y).all() and ((y>=0)&(y<=60)).all()\n    REF[\'check_baselines\'](times,y,60,baseline,config)\n    masks=[REF[\'partition_masks\'](times,60,*fold) for fold in config[\'folds\']]\n    archive=None; checksums=None\n    class BudgetStore:\n        def read(self,name): return store.read(name)\n        def write(self,name,data,mime=\'application/json\'):\n            store.write(name,data,mime)\n            if time.monotonic()>=deadline: raise BudgetStop(\'Saved completed month; session budget reached\')\n    fit_workers=min(2,joblib.cpu_count())\n    families=sorted({f.split(\'__\')[1] for f in pending},key=lambda f:(f!=\'flow\',f))\n    save_reports()\n    for family in families:\n        if time.monotonic()>=deadline: break\n        columns=[f for f in config[\'candidate_features\'] if f.split(\'__\')[1]==family]\n        xid=dict(type=\'f4z_research_cache_v1\',source=config[\'audit_sources\'][\'X\'],columns=columns,\n                 origin_end=config[\'origin_end_exclusive\'],cadence=30,minimum_baseline_n=1000,allowed_source_status=[\'PROVISIONAL\'])\n        print(f\'Loading family {family}; completed caches are reused.\',flush=True)\n        xm,xt,xx=REF[\'read_cache\'](store,f\'SCREEN_X_F4Z_{family}.npz\',xid)\n        if xm[\'completed\']!=config[\'months\']:\n            if archive is None:\n                archive=REF[\'RemoteArchive\'](credentials,config[\'audit_sources\'][\'X\'])\n                checksums={}\n                for line in archive.read(\'FILES.sha256\').decode().splitlines():\n                    if line.strip(): h,name=line.split(maxsplit=1); checksums[name.lstrip(\'*\')]=h\n            try:\n                xt,xx,xhash=REF[\'load_family\'](archive,BudgetStore(),family,columns,config[\'audit_sources\'][\'X\'],checksums,config,min(2,joblib.cpu_count()))\n            except BudgetStop:\n                break\n        else:\n            xt=pd.DatetimeIndex(xt,tz=\'UTC\'); xhash=xm[\'array_sha256\']\n        assert xt.is_unique and xt.is_monotonic_increasing and (xt<pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n        index=xt.get_indexer(times); assert (index>=0).all()\n        names=iter(f for f in pending if f.split(\'__\')[1]==family)\n        writer=OPT[\'OrderedWriter\'](write_scores.write,capacity=2)\n        active={}; finished=[]\n        try:\n            with threadpool_limits(limits=1),ThreadPoolExecutor(max_workers=fit_workers) as pool:\n                def submit_next():\n                    if time.monotonic()>=deadline: return False\n                    feature=next(names,None)\n                    if feature is None: return False\n                    filename=\'job_\'+REF[\'sha\']((feature+\'|60\').encode())[:24]+\'.json\'\n                    raw=read_scores.read(filename)\n                    job=unpack(raw,sig,feature,xhash,yhash) if raw else {\'screen_signature\':sig,\'feature\':feature,\'horizon_minutes\':60,\n                        \'X_cache_sha256\':xhash,\'Y_cache_sha256\':yhash,\'fits\':{},\'folds\':{},\'matrix\':{}}\n                    def checkpoint(value): writer.submit(filename,pack(value))\n                    future=pool.submit(fit_job,job,xx[index,columns.index(feature)],times,y,masks,config,checkpoint,deadline)\n                    active[future]=(feature,filename); return True\n                for _ in range(fit_workers): submit_next()\n                while active:\n                    done,_=wait(active,timeout=20,return_when=FIRST_COMPLETED)\n                    if not done: print(f\'{family}: {len(active)} active features; elapsed {(time.monotonic()-start)/60:.1f} minutes\',flush=True)\n                    for future in done:\n                        feature,filename=active.pop(future); job=future.result()\n                        all_jobs[feature+\'|60\']=job; finished.append((feature,filename,job))\n                        submit_next()\n        finally:\n            writer.close()  # Failure must propagate; no completed-job promotion before flush.\n        # Read back durable snapshots before recording completion in the aggregate.\n        for feature,filename,job in finished:\n            read_scores.ids.pop(filename,None)\n            raw=read_scores.read(filename)\n            if raw is None: continue  # Budget ended before first fit.\n            stored=unpack(raw,sig,feature,xhash,yhash)\n            assert REF[\'signature\'](stored)==REF[\'signature\'](job)\n            state[\'jobs\'][feature+\'|60\']={\'file_id\':read_scores.ids[filename],\'sha256\':REF[\'signature\'](stored),\n                                         \'complete_folds\':len(stored[\'folds\'])}\n        state[\'status\']=\'PARTIAL_1h_ONLY\'; store.json(\'SCREEN_1h_RESULTS.json\',state); save_reports()\n    complete=sum(len(all_jobs.get(f+\'|60\',{}).get(\'folds\',{}))==5 for f in config[\'candidate_features\'])\n    state[\'status\']=\'COMPLETE_1h_ONLY\' if complete==len(config[\'candidate_features\']) else \'PAUSED_SESSION_BUDGET_1h_ONLY\'\n    state[\'completed_features\']=complete; state[\'session_seconds\']=time.monotonic()-start\n    for k,file_id in config[\'source_ids\'].items():\n        assert service.files().get(fileId=file_id,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()==config[\'audit_sources\'][k]\n    store.json(\'SCREEN_1h_RESULTS.json\',state); save_reports()\n    print(f\'{complete}/{len(config["candidate_features"])} complete. Rerun this same notebook to resume 1h only.\',flush=True)\n', 'io_guard': '"""Session-local Drive transport pacing; no model/config/checkpoint changes."""\nimport json\nimport random\nimport threading\nimport time\nfrom urllib.parse import urlsplit\nfrom email.utils import parsedate_to_datetime\n\n\ndef is_drive(url):\n    parsed=urlsplit(str(url))\n    return parsed.hostname==\'drive.googleapis.com\' or (\n        parsed.hostname==\'www.googleapis.com\' and parsed.path.startswith((\'/drive/\',\'/upload/drive/\')))\n\n\ndef limited(status, content):\n    if int(status)==429: return True\n    if int(status)!=403: return False\n    try:\n        value=json.loads(content)\n        reasons={e.get(\'reason\') for e in value.get(\'error\',{}).get(\'errors\',[])}\n        return bool(reasons & {\'rateLimitExceeded\',\'userRateLimitExceeded\'})\n    except (ValueError,TypeError,AttributeError):\n        return False\n\n\nclass Gate:\n    def __init__(self, interval=3.0, clock=time.monotonic, sleep=time.sleep):\n        self.interval,self.clock,self.sleep=interval,clock,sleep\n        self.next_allowed=0.0; self.lock=threading.Lock()\n    def take(self):\n        while True:\n            with self.lock:\n                now=self.clock(); wait=self.next_allowed-now\n                if wait<=0:\n                    self.next_allowed=now+self.interval\n                    return\n            self.sleep(min(wait,60))\n    def defer(self, seconds):\n        with self.lock: self.next_allowed=max(self.next_allowed,self.clock()+seconds)\n\n\ndef retry_delay(attempt, headers):\n    delay=min(2**(attempt+1)+random.random(),60)\n    value=headers.get(\'retry-after\') or headers.get(\'Retry-After\')\n    if value:\n        try: delay=max(delay,float(value))\n        except (ValueError,TypeError):\n            try: delay=max(delay,parsedate_to_datetime(value).timestamp()-time.time())\n            except (ValueError,TypeError,OverflowError): pass\n    return max(0,delay)\n\n\ndef run_request(send, decode, gate, body=None, max_retries=8):\n    """Retry explicit rejection only. Ambiguous POST failures are never replayed."""\n    offset=None\n    if hasattr(body,\'read\'):\n        try: offset=body.tell()\n        except (AttributeError,OSError): pass\n    for attempt in range(max_retries+1):\n        if attempt and hasattr(body,\'read\'):\n            if offset is None: raise RuntimeError(\'Cannot safely replay a streaming Drive request\')\n            body.seek(offset)\n        gate.take()\n        response=send()  # Exceptions and ambiguous network failures propagate unchanged.\n        status,content,headers,close=decode(response)\n        if not limited(status,content) or attempt==max_retries:\n            if limited(status,content): print(\'Drive rate limit persisted after bounded retries; saved checkpoints remain reusable.\',flush=True)\n            return response\n        delay=retry_delay(attempt,headers)\n        close()\n        gate.defer(delay)\n        print(f\'Drive rate limit: retry {attempt+1}/{max_retries}, waiting at least {delay:.1f}s. Fit settings unchanged.\',flush=True)\n\n\ndef install(interval=3.0):\n    import google_auth_httplib2\n    from google.auth.transport.requests import AuthorizedSession\n    marker=\'_scalewave_drive_rate_guard_v1\'\n    cls=google_auth_httplib2.AuthorizedHttp\n    if getattr(cls,marker,False):\n        print(\'Drive request guard already installed in this runtime.\',flush=True)\n        return\n    gate=Gate(interval)\n    original_http=cls.request\n    def http_request(self, uri, method=\'GET\', body=None, headers=None, *args, **kwargs):\n        send=lambda:original_http(self,uri,method,body,headers,*args,**kwargs)\n        if not is_drive(uri): return send()\n        def decode(response):\n            metadata,content=response\n            return int(metadata.status),content,metadata,lambda:None\n        return run_request(send,decode,gate,body=body)\n    original_session=AuthorizedSession.request\n    def session_request(self, method, url, *args, **kwargs):\n        send=lambda:original_session(self,method,url,*args,**kwargs)\n        if not is_drive(url): return send()\n        def decode(response):\n            # Do not consume successful streaming ZIP range responses.\n            content=response.content if response.status_code in (403,429) else b\'\'\n            return response.status_code,content,response.headers,response.close\n        return run_request(send,decode,gate,body=kwargs.get(\'data\'))\n    cls.request=http_request\n    AuthorizedSession.request=session_request\n    setattr(cls,marker,True)\n    print(f\'Drive I/O guard active: requests spaced by {interval:g}s per runtime; rate-limit retries enabled.\',flush=True)\n', 'source_compat': '"""Narrow, recorded compatibility patch for Drive modifiedTime-only changes.\n\nOriginal runner sources and checkpoint identities stay immutable. Only their two\narchive-metadata assertions are replaced; executable patch hashes are recorded.\n"""\nimport hashlib\nimport json\n\nAPPROVED = {\n    \'15m\':\'38d9944751c3ad2815a39024b40e3427d113a3fd7ada1acf521d72fdcc2f9777\',\n    \'1h\':\'017f7326c9c5a43b71a3aede7b2a10fe2428c08efdd7d18aa1fa86f7241202eb\',\n}\nOLD = "assert service.files().get(fileId=file_id,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()==config[\'audit_sources\'][k]"\nNEW = "assert SOURCE_METADATA_CHECK(service,config,k,file_id)"\n\n\ndef compare(actual, expected):\n    assert set(actual)==set(expected), \'Source metadata fields changed\'\n    assert actual.get(\'md5Checksum\') and expected.get(\'md5Checksum\'), \'Source checksum missing\'\n    different=[key for key in expected if actual[key]!=expected[key]]\n    assert set(different)<= {\'modifiedTime\'}, \'Source identity/content changed: \'+\', \'.join(different)\n    return different\n\n\ndef compile_runner(source, namespace, horizon, patch_hash):\n    original_hash=hashlib.sha256(source.encode()).hexdigest()\n    assert original_hash==APPROVED[horizon], \'Runner not approved for metadata-only compatibility\'\n    assert source.count(OLD)==2, \'Expected exactly the startup and final source checks\'\n    patched=source.replace(OLD,NEW)\n    evidence={\'policy\':\'Accept modifiedTime-only difference; all other returned fields, including size and MD5, must match.\',\n              \'original_runner_sha256\':original_hash,\n              \'executed_runner_sha256\':hashlib.sha256(patched.encode()).hexdigest(),\n              \'compat_module_sha256\':patch_hash,\'horizon\':horizon,\'sources\':{}}\n    def verify(service,config,key,file_id):\n        actual=service.files().get(fileId=file_id,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()\n        expected=config[\'audit_sources\'][key]\n        differences=compare(actual,expected)\n        evidence[\'sources\'][key]={\'expected\':expected,\'observed\':actual,\'accepted_differences\':differences,\n                                  \'id_name_size_md5_match\':True}\n        # Separate audit receipt; never rewrite a source archive, frozen contract,\n        # old model receipt, or cache identity to conceal its original metadata.\n        store=namespace[\'REF\'][\'DriveStore\'](service,config[\'folder\'])\n        store.json(f\'SOURCE_METADATA_COMPAT_{horizon}.json\',evidence)\n        if differences:\n            print(f\'{key}: Drive modification time changed; file ID, name, size and MD5 still match. Recorded metadata-only compatibility.\',flush=True)\n        return True\n    namespace[\'SOURCE_METADATA_CHECK\']=verify\n    exec(compile(patched,f\'screen_{horizon}_runner.py\',\'exec\'),namespace)\n    return evidence\n', 'range_reader': "import io\nMAX_TRANSFER = 12 * 1024**3\nclass DriveRangeReader(io.RawIOBase):\n    def __init__(self, session, file_id, size):\n        self.session, self.file_id, self.size = session, file_id, size\n        self.pos = self.transferred = 0\n\n    def readable(self): return True\n    def seekable(self): return True\n    def tell(self): return self.pos\n\n    def seek(self, offset, whence=0):\n        target = offset if whence == 0 else self.pos+offset if whence == 1 else self.size+offset\n        if whence not in (0, 1, 2) or target < 0:\n            raise ValueError('Invalid seek')\n        self.pos = target\n        return target\n\n    def read(self, n=-1):\n        n = min(self.size-self.pos, n if n >= 0 else self.size-self.pos)\n        if n <= 0: return b''\n        if n > 32*1024**2 or self.transferred+n > MAX_TRANSFER:\n            raise RuntimeError('Remote read budget reached; no full download attempted.')\n        start, end = self.pos, self.pos+n-1\n        url = f'https://www.googleapis.com/drive/v3/files/{self.file_id}'\n        with self.session.get(url, params={'alt':'media'},\n                              headers={'Range':f'bytes={start}-{end}'}, stream=True, timeout=90) as response:\n            if response.status_code != 206:\n                raise RuntimeError(f'Expected partial response 206, got {response.status_code}; stopped without reading full body.')\n            if response.headers.get('Content-Range') != f'bytes {start}-{end}/{self.size}':\n                raise RuntimeError('Unexpected range response')\n            data = response.raw.read(n+1)\n            if len(data) != n: raise RuntimeError('Incomplete or oversized range')\n        self.pos += n\n        self.transferred += n\n        return data\n\n"}
hashes = {'reference': 'e584210af02adb15168245cd13e2cb6603eed882069cb75cd877d29122a6821b', 'optimized': '29fd33d96055bc3fe20829806211d34333ec4e0be12fb676c9bb37d4d0f5ed7b', 'report': '2a557e64af72c9e9c94bdf6faad0c94879c456c62273f3ca37702b7601fd3726', 'runner': '017f7326c9c5a43b71a3aede7b2a10fe2428c08efdd7d18aa1fa86f7241202eb', 'io_guard': '3847bcce7a3adeb864718bc1ae91c2a0d2c6c99f39e59b1f6d09fc885dec6285', 'source_compat': 'e8d85cc0d35be2b1a43cf5b7cb774d41d0f5d5a34e22e28ade5367e0153fb303', 'range_reader': '4ac4e8be2378643f90af71c3ff286cdcca65921277a959d99a3cecad4be6bfcb'}
assert all(hashlib.sha256(v.encode()).hexdigest()==hashes[k] for k,v in sources.items())
IO_GUARD = {}
exec(compile(sources["io_guard"], "drive_rate_guard.py", "exec"), IO_GUARD)
IO_GUARD["install"]()
REF = {}
exec(compile(sources["range_reader"], "range_reader.py", "exec"), REF)
exec(compile(sources["reference"], "screen_runner.py", "exec"), REF)
OPT = {"REF":REF}
exec(compile(sources["optimized"], "screen_throughput.py", "exec"), OPT)
REPORT = {}
exec(compile(sources["report"], "timing_results_report.py", "exec"), REPORT)
RUNNER = {"REF":REF,"OPT":OPT,"REPORT":REPORT}
SOURCE_COMPAT = {}
exec(compile(sources["source_compat"], "source_metadata_compat.py", "exec"), SOURCE_COMPAT)
SOURCE_COMPAT["compile_runner"](sources["runner"], RUNNER, "1h", hashes["source_compat"])
config = json.loads('{"version": "btc_time_high_single_feature_screen_v1", "symbol": "BTCUSDT", "target": "time_to_high_min", "folder": "1DXhoOLybfhEIt3phvPfG3cL3M31FPFEC", "source_ids": {"X": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "Y": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM"}, "audit_sources": {"X": {"id": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "name": "Handoff 1_F4_f4Z.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "043b2f570cfb63215749385d72c7fe59", "size": "15600742757"}, "Y": {"id": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM", "name": "HANDOFF_2_CANONICAL_XY_V1_1.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "bf2cb8dd274222747fde1b2e979b42b7", "size": "3033662926"}}, "audit_receipt_sha256": "6daca0c3d14b09de5695cd83cc3e86418f39d8afff1f4116611c0ed7cf5d6937", "baseline_signature": "729209d72e1ee50796f4964f087eb66b7e4fb87a6200e926cf8ef838af92272e", "candidate_features": ["F4Z__basis__basis_change_percentile__12h__value", "F4Z__basis__basis_change_percentile__15m__value", "F4Z__basis__basis_change_percentile__1h__value", "F4Z__basis__basis_change_percentile__4h__value", "F4Z__basis__basis_change_percentile__5m__value", "F4Z__basis__basis_change_rz__12h__value", "F4Z__basis__basis_change_rz__15m__value", "F4Z__basis__basis_change_rz__1h__value", "F4Z__basis__basis_change_rz__4h__value", "F4Z__basis__basis_change_rz__5m__value", "F4Z__basis__basis_level_percentile__12h__value", "F4Z__basis__basis_level_percentile__15m__value", "F4Z__basis__basis_level_percentile__1h__value", "F4Z__basis__basis_level_percentile__4h__value", "F4Z__basis__basis_level_percentile__5m__value", "F4Z__basis__basis_level_rz__12h__value", "F4Z__basis__basis_level_rz__15m__value", "F4Z__basis__basis_level_rz__1h__value", "F4Z__basis__basis_level_rz__4h__value", "F4Z__basis__basis_level_rz__5m__value", "F4Z__basis__basis_path_retention_percentile__12h__value", "F4Z__basis__basis_path_retention_percentile__15m__value", "F4Z__basis__basis_path_retention_percentile__1h__value", "F4Z__basis__basis_path_retention_percentile__4h__value", "F4Z__basis__basis_path_retention_percentile__5m__value", "F4Z__basis__basis_persistence_percentile__12h__value", "F4Z__basis__basis_persistence_percentile__15m__value", "F4Z__basis__basis_persistence_percentile__1h__value", "F4Z__basis__basis_persistence_percentile__4h__value", "F4Z__basis__basis_persistence_percentile__5m__value", "F4Z__basis__basis_range_percentile__12h__value", "F4Z__basis__basis_range_percentile__15m__value", "F4Z__basis__basis_range_percentile__1h__value", "F4Z__basis__basis_range_percentile__4h__value", "F4Z__basis__basis_range_percentile__5m__value", "F4Z__basis__basis_range_rz__12h__value", "F4Z__basis__basis_range_rz__15m__value", "F4Z__basis__basis_range_rz__1h__value", "F4Z__basis__basis_range_rz__4h__value", "F4Z__basis__basis_range_rz__5m__value", "F4Z__flow__cvd_path_efficiency_percentile__12h__value", "F4Z__flow__cvd_path_efficiency_percentile__15m__value", "F4Z__flow__cvd_path_efficiency_percentile__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value", "F4Z__flow__cvd_path_efficiency_percentile__5m__value", "F4Z__flow__flow_acceleration_percentile__15m__12h__value", "F4Z__flow__flow_acceleration_percentile__15m__1h__value", "F4Z__flow__flow_acceleration_percentile__15m__4h__value", "F4Z__flow__flow_acceleration_percentile__1h__12h__value", "F4Z__flow__flow_acceleration_percentile__1h__4h__value", "F4Z__flow__flow_acceleration_percentile__4h__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__15m__value", "F4Z__flow__flow_acceleration_percentile__5m__1h__value", "F4Z__flow__flow_acceleration_percentile__5m__4h__value", "F4Z__flow__flow_acceleration_rz__15m__12h__value", "F4Z__flow__flow_acceleration_rz__15m__1h__value", "F4Z__flow__flow_acceleration_rz__15m__4h__value", "F4Z__flow__flow_acceleration_rz__1h__12h__value", "F4Z__flow__flow_acceleration_rz__1h__4h__value", "F4Z__flow__flow_acceleration_rz__4h__12h__value", "F4Z__flow__flow_acceleration_rz__5m__12h__value", "F4Z__flow__flow_acceleration_rz__5m__15m__value", "F4Z__flow__flow_acceleration_rz__5m__1h__value", "F4Z__flow__flow_acceleration_rz__5m__4h__value", "F4Z__flow__flow_activity_percentile__12h__value", "F4Z__flow__flow_activity_percentile__15m__value", "F4Z__flow__flow_activity_percentile__1h__value", "F4Z__flow__flow_activity_percentile__4h__value", "F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_activity_rz__12h__value", "F4Z__flow__flow_activity_rz__15m__value", "F4Z__flow__flow_activity_rz__1h__value", "F4Z__flow__flow_activity_rz__4h__value", "F4Z__flow__flow_activity_rz__5m__value", "F4Z__flow__flow_confirmation_percentile__12h__value", "F4Z__flow__flow_confirmation_percentile__15m__value", "F4Z__flow__flow_confirmation_percentile__1h__value", "F4Z__flow__flow_confirmation_percentile__4h__value", "F4Z__flow__flow_confirmation_percentile__5m__value", "F4Z__flow__flow_confirmation_rz__12h__value", "F4Z__flow__flow_confirmation_rz__15m__value", "F4Z__flow__flow_confirmation_rz__1h__value", "F4Z__flow__flow_confirmation_rz__4h__value", "F4Z__flow__flow_confirmation_rz__5m__value", "F4Z__flow__flow_cvd_slope_percentile__12h__value", "F4Z__flow__flow_cvd_slope_percentile__15m__value", "F4Z__flow__flow_cvd_slope_percentile__1h__value", "F4Z__flow__flow_cvd_slope_percentile__4h__value", "F4Z__flow__flow_cvd_slope_percentile__5m__value", "F4Z__flow__flow_cvd_slope_rz__12h__value", "F4Z__flow__flow_cvd_slope_rz__15m__value", "F4Z__flow__flow_cvd_slope_rz__1h__value", "F4Z__flow__flow_cvd_slope_rz__4h__value", "F4Z__flow__flow_cvd_slope_rz__5m__value", "F4Z__flow__flow_pressure_percentile__12h__value", "F4Z__flow__flow_pressure_percentile__15m__value", "F4Z__flow__flow_pressure_percentile__1h__value", "F4Z__flow__flow_pressure_percentile__4h__value", "F4Z__flow__flow_pressure_percentile__5m__value", "F4Z__flow__flow_pressure_rz__12h__value", "F4Z__flow__flow_pressure_rz__15m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__flow_pressure_rz__4h__value", "F4Z__flow__flow_pressure_rz__5m__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__4h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__15m__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__4h__value", "F4Z__flow__signed_flow_acceleration_rz__4h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__15m__value", "F4Z__flow__signed_flow_acceleration_rz__5m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__4h__value", "F4Z__funding__funding_change_percentile__15m__value", "F4Z__funding__funding_change_percentile__1h__value", "F4Z__funding__funding_change_percentile__4h__value", "F4Z__funding__funding_change_percentile__5m__value", "F4Z__funding__funding_change_rz__15m__value", "F4Z__funding__funding_change_rz__1h__value", "F4Z__funding__funding_change_rz__4h__value", "F4Z__funding__funding_change_rz__5m__value", "F4Z__funding__funding_level_percentile__15m__value", "F4Z__funding__funding_level_percentile__1h__value", "F4Z__funding__funding_level_percentile__4h__value", "F4Z__funding__funding_level_percentile__5m__value", "F4Z__funding__funding_level_rz__15m__value", "F4Z__funding__funding_level_rz__1h__value", "F4Z__funding__funding_level_rz__4h__value", "F4Z__funding__funding_level_rz__5m__value", "F4Z__funding__funding_path_retention_percentile__15m__value", "F4Z__funding__funding_path_retention_percentile__1h__value", "F4Z__funding__funding_path_retention_percentile__4h__value", "F4Z__funding__funding_path_retention_percentile__5m__value", "F4Z__funding__funding_persistence_percentile__15m__value", "F4Z__funding__funding_persistence_percentile__1h__value", "F4Z__funding__funding_persistence_percentile__4h__value", "F4Z__funding__funding_persistence_percentile__5m__value", "F4Z__funding__funding_range_percentile__15m__value", "F4Z__funding__funding_range_percentile__1h__value", "F4Z__funding__funding_range_percentile__4h__value", "F4Z__funding__funding_range_percentile__5m__value", "F4Z__funding__funding_range_rz__15m__value", "F4Z__funding__funding_range_rz__1h__value", "F4Z__funding__funding_range_rz__4h__value", "F4Z__funding__funding_range_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__5m__value", "F4Z__liquidation__liq_acceleration_percentile__15m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__4h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__4h__value", "F4Z__liquidation__liq_acceleration_percentile__4h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__15m__value", "F4Z__liquidation__liq_acceleration_percentile__5m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__4h__value", "F4Z__liquidation__liq_acceleration_rz__15m__12h__value", "F4Z__liquidation__liq_acceleration_rz__15m__1h__value", "F4Z__liquidation__liq_acceleration_rz__15m__4h__value", "F4Z__liquidation__liq_acceleration_rz__1h__12h__value", "F4Z__liquidation__liq_acceleration_rz__1h__4h__value", "F4Z__liquidation__liq_acceleration_rz__4h__12h__value", "F4Z__liquidation__liq_acceleration_rz__5m__15m__value", "F4Z__liquidation__liq_acceleration_rz__5m__1h__value", "F4Z__liquidation__liq_acceleration_rz__5m__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__5m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__5m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__15m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__1h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__4h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__5m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__12h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__15m__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__1h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_percentile__12h__value", "F4Z__liquidation__liq_side_pressure_percentile__15m__value", "F4Z__liquidation__liq_side_pressure_percentile__1h__value", "F4Z__liquidation__liq_side_pressure_percentile__4h__value", "F4Z__liquidation__liq_side_pressure_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_rz__12h__value", "F4Z__liquidation__liq_side_pressure_rz__15m__value", "F4Z__liquidation__liq_side_pressure_rz__1h__value", "F4Z__liquidation__liq_side_pressure_rz__4h__value", "F4Z__liquidation__liq_side_pressure_rz__5m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__12h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__15m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__1h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__4h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__5m__value", "F4Z__liquidation__venue_breadth_percentile__12h__value", "F4Z__liquidation__venue_breadth_percentile__15m__value", "F4Z__liquidation__venue_breadth_percentile__1h__value", "F4Z__liquidation__venue_breadth_percentile__4h__value", "F4Z__liquidation__venue_breadth_percentile__5m__value", "F4Z__liquidation__venue_fragmentation_percentile__12h__value", "F4Z__liquidation__venue_fragmentation_percentile__15m__value", "F4Z__liquidation__venue_fragmentation_percentile__1h__value", "F4Z__liquidation__venue_fragmentation_percentile__4h__value", "F4Z__liquidation__venue_fragmentation_percentile__5m__value", "F4Z__oi__leverage_unwind_percentile__12h__value", "F4Z__oi__leverage_unwind_percentile__15m__value", "F4Z__oi__leverage_unwind_percentile__1h__value", "F4Z__oi__leverage_unwind_percentile__4h__value", "F4Z__oi__leverage_unwind_percentile__5m__value", "F4Z__oi__leverage_unwind_rz__12h__value", "F4Z__oi__leverage_unwind_rz__15m__value", "F4Z__oi__leverage_unwind_rz__1h__value", "F4Z__oi__leverage_unwind_rz__4h__value", "F4Z__oi__leverage_unwind_rz__5m__value", "F4Z__oi__oi_extreme_recency_percentile__12h__value", "F4Z__oi__oi_extreme_recency_percentile__15m__value", "F4Z__oi__oi_extreme_recency_percentile__1h__value", "F4Z__oi__oi_extreme_recency_percentile__4h__value", "F4Z__oi__oi_extreme_recency_percentile__5m__value", "F4Z__oi__oi_path_retention_percentile__12h__value", "F4Z__oi__oi_path_retention_percentile__15m__value", "F4Z__oi__oi_path_retention_percentile__1h__value", "F4Z__oi__oi_path_retention_percentile__4h__value", "F4Z__oi__oi_path_retention_percentile__5m__value", "F4Z__oi__oi_range_position_percentile__12h__value", "F4Z__oi__oi_range_position_percentile__15m__value", "F4Z__oi__oi_range_position_percentile__1h__value", "F4Z__oi__oi_range_position_percentile__4h__value", "F4Z__oi__oi_range_position_percentile__5m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__12h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__15m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__1h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__4h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__5m__value", "F4Z__price__price_acceleration_percentile__15m__12h__value", "F4Z__price__price_acceleration_percentile__15m__1h__value", "F4Z__price__price_acceleration_percentile__15m__4h__value", "F4Z__price__price_acceleration_percentile__1h__12h__value", "F4Z__price__price_acceleration_percentile__1h__4h__value", "F4Z__price__price_acceleration_percentile__4h__12h__value", "F4Z__price__price_acceleration_percentile__5m__12h__value", "F4Z__price__price_acceleration_percentile__5m__15m__value", "F4Z__price__price_acceleration_percentile__5m__1h__value", "F4Z__price__price_acceleration_percentile__5m__4h__value", "F4Z__price__price_acceleration_rz__15m__12h__value", "F4Z__price__price_acceleration_rz__15m__1h__value", "F4Z__price__price_acceleration_rz__15m__4h__value", "F4Z__price__price_acceleration_rz__1h__12h__value", "F4Z__price__price_acceleration_rz__1h__4h__value", "F4Z__price__price_acceleration_rz__4h__12h__value", "F4Z__price__price_acceleration_rz__5m__12h__value", "F4Z__price__price_acceleration_rz__5m__15m__value", "F4Z__price__price_acceleration_rz__5m__1h__value", "F4Z__price__price_acceleration_rz__5m__4h__value", "F4Z__price__price_displacement_percentile__12h__value", "F4Z__price__price_displacement_percentile__15m__value", "F4Z__price__price_displacement_percentile__1h__value", "F4Z__price__price_displacement_percentile__4h__value", "F4Z__price__price_displacement_percentile__5m__value", "F4Z__price__price_displacement_rz__12h__value", "F4Z__price__price_displacement_rz__15m__value", "F4Z__price__price_displacement_rz__1h__value", "F4Z__price__price_displacement_rz__4h__value", "F4Z__price__price_displacement_rz__5m__value", "F4Z__price__price_excursion_range_percentile__12h__value", "F4Z__price__price_excursion_range_percentile__15m__value", "F4Z__price__price_excursion_range_percentile__1h__value", "F4Z__price__price_excursion_range_percentile__4h__value", "F4Z__price__price_excursion_range_percentile__5m__value", "F4Z__price__price_excursion_range_rz__12h__value", "F4Z__price__price_excursion_range_rz__15m__value", "F4Z__price__price_excursion_range_rz__1h__value", "F4Z__price__price_excursion_range_rz__4h__value", "F4Z__price__price_excursion_range_rz__5m__value", "F4Z__price__price_move_retention_percentile__12h__value", "F4Z__price__price_move_retention_percentile__15m__value", "F4Z__price__price_move_retention_percentile__1h__value", "F4Z__price__price_move_retention_percentile__4h__value", "F4Z__price__price_move_retention_percentile__5m__value", "F4Z__volatility__range_expansion_percentile__15m__12h__value", "F4Z__volatility__range_expansion_percentile__15m__1h__value", "F4Z__volatility__range_expansion_percentile__15m__4h__value", "F4Z__volatility__range_expansion_percentile__1h__12h__value", "F4Z__volatility__range_expansion_percentile__1h__4h__value", "F4Z__volatility__range_expansion_percentile__4h__12h__value", "F4Z__volatility__range_expansion_percentile__5m__12h__value", "F4Z__volatility__range_expansion_percentile__5m__15m__value", "F4Z__volatility__range_expansion_percentile__5m__1h__value", "F4Z__volatility__range_expansion_percentile__5m__4h__value", "F4Z__volatility__range_expansion_rz__15m__12h__value", "F4Z__volatility__range_expansion_rz__15m__1h__value", "F4Z__volatility__range_expansion_rz__15m__4h__value", "F4Z__volatility__range_expansion_rz__1h__12h__value", "F4Z__volatility__range_expansion_rz__1h__4h__value", "F4Z__volatility__range_expansion_rz__4h__12h__value", "F4Z__volatility__range_expansion_rz__5m__12h__value", "F4Z__volatility__range_expansion_rz__5m__15m__value", "F4Z__volatility__range_expansion_rz__5m__1h__value", "F4Z__volatility__range_expansion_rz__5m__4h__value", "F4Z__volatility__realized_variance_level_percentile__12h__value", "F4Z__volatility__realized_variance_level_percentile__15m__value", "F4Z__volatility__realized_variance_level_percentile__1h__value", "F4Z__volatility__realized_variance_level_percentile__4h__value", "F4Z__volatility__realized_variance_level_percentile__5m__value", "F4Z__volatility__realized_variance_level_rz__12h__value", "F4Z__volatility__realized_variance_level_rz__15m__value", "F4Z__volatility__realized_variance_level_rz__1h__value", "F4Z__volatility__realized_variance_level_rz__4h__value", "F4Z__volatility__realized_variance_level_rz__5m__value", "F4Z__volatility__vol_acceleration_percentile__15m__12h__value", "F4Z__volatility__vol_acceleration_percentile__15m__1h__value", "F4Z__volatility__vol_acceleration_percentile__15m__4h__value", "F4Z__volatility__vol_acceleration_percentile__1h__12h__value", "F4Z__volatility__vol_acceleration_percentile__1h__4h__value", "F4Z__volatility__vol_acceleration_percentile__4h__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__15m__value", "F4Z__volatility__vol_acceleration_percentile__5m__1h__value", "F4Z__volatility__vol_acceleration_percentile__5m__4h__value", "F4Z__volatility__vol_acceleration_rz__15m__12h__value", "F4Z__volatility__vol_acceleration_rz__15m__1h__value", "F4Z__volatility__vol_acceleration_rz__15m__4h__value", "F4Z__volatility__vol_acceleration_rz__1h__12h__value", "F4Z__volatility__vol_acceleration_rz__1h__4h__value", "F4Z__volatility__vol_acceleration_rz__4h__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__15m__value", "F4Z__volatility__vol_acceleration_rz__5m__1h__value", "F4Z__volatility__vol_acceleration_rz__5m__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__12h__value", "F4Z__volatility__vol_shock_concentration_percentile__15m__value", "F4Z__volatility__vol_shock_concentration_percentile__1h__value", "F4Z__volatility__vol_shock_concentration_percentile__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__5m__value", "F4Z__volatility__vol_shock_recency_percentile__12h__value", "F4Z__volatility__vol_shock_recency_percentile__15m__value", "F4Z__volatility__vol_shock_recency_percentile__1h__value", "F4Z__volatility__vol_shock_recency_percentile__4h__value", "F4Z__volatility__vol_shock_recency_percentile__5m__value"], "pilot_features": ["F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value"], "pilot_horizons": [15, 2880, 18000], "horizons": [15, 60, 240, 480, 960, 2880, 4320, 7200, 12000, 18000], "months": ["2022-04", "2022-05", "2022-06", "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12", "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06", "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"], "origin_end_exclusive": "2024-07-01T00:00:00Z", "reserve_start": "2024-07-24T00:01:00Z", "cadence_minutes": 30, "folds": [["2023-04-01", "2023-07-01"], ["2023-07-01", "2023-10-01"], ["2023-10-01", "2024-01-01"], ["2024-01-01", "2024-04-01"], ["2024-04-01", "2024-07-01"]], "grid": [[2, 64], [2, 256], [3, 64], [3, 256], [5, 64], [5, 256]], "minimum_observed_training": 1000, "inner_validation": "Preceding three calendar months; inner training label_end < inner start, inner validation label_end < outer start. Outer training label_end < outer start.", "model": "Single raw F4Z input plus explicit missing indicator; DecisionTreeRegressor absolute_error, seed48, depth/leaf grid only.", "preprocessing": "Train-only finite-value median imputation plus missing indicator. No feature scaling. Baseline_n >=1000, finite, valid, recorded available_after <= origin, scientific_status PROVISIONAL explicitly allowed for provisional research only.", "selection": "Lowest inner MAE; ties shallower depth then larger leaf. Never use outer MAE to choose fold settings.", "ineligible_policy": "If fewer than1000 observed training values or constant observed training feature: no fit; if no eligible inner setting or outer training insufficient, predict outer-training target median on all outer rows. Report fallback and coverage.", "evaluation": "Exact original baseline rows; primary MAE minutes, secondary percent of horizon; pooled row-weighted errors; compare all three baselines. Record missing/observed subset errors. No significance/global optimum/trading claim.", "pilot_scope": "3 flow features x3 horizons x5 folds x(6 inner+1 outer) = at most315 fits. Features selected for differing market meanings and sampled coverage, not outcome scores.", "pilot_meanings": {"F4Z__flow__flow_activity_percentile__5m__value": "Recent flow activity relative to history", "F4Z__flow__flow_pressure_rz__1h__value": "Directional trading pressure relative to history", "F4Z__flow__cvd_path_efficiency_percentile__4h__value": "How consistently cumulative trading flow follows one direction"}, "cache_policy": "One compact research-only NPZ per horizon/family in Drive, no PC cache. Float64 values preserve source conversion. Checkpoint each month; immutable signature/hash checks. Family cache includes all candidate columns from that family (85 flow) to avoid rereading later. <=128MiB per cache.", "checkpoint_policy": "Per-fit compact JSON scores in Timing_Screen_Fit_Scores; no model binaries or row predictions. Partial family/horizon data resume completed months. An interrupted unsaved fit may restart. Pilot completed fits reused by full mode under identical identity/versions.", "workers": "Up to4 bounded I/O workers; up to2 fit workers; native thread pools limited. No automatic full run after pilot.", "limitations": ["F4Z upstream timing/normalization remains unresolved; PROVISIONAL research only.", "Sample-based candidate preselection does not prove excluded features useless across full history.", "Single-feature trees do not establish globally best features or combinations.", "Reserved period previously exposed in other project work but excluded from this task.", "Errors overlap in time; no iid significance assumptions."], "full_launch_policy": "Review pilot correctness, runtime, memory and results before authorizing full386-feature expansion. Do not change to full mode yet."}')
pilot_identity = json.loads('{"config": {"version": "btc_time_high_single_feature_screen_v1", "symbol": "BTCUSDT", "target": "time_to_high_min", "folder": "1DXhoOLybfhEIt3phvPfG3cL3M31FPFEC", "source_ids": {"X": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "Y": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM"}, "audit_sources": {"X": {"id": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "name": "Handoff 1_F4_f4Z.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "043b2f570cfb63215749385d72c7fe59", "size": "15600742757"}, "Y": {"id": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM", "name": "HANDOFF_2_CANONICAL_XY_V1_1.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "bf2cb8dd274222747fde1b2e979b42b7", "size": "3033662926"}}, "audit_receipt_sha256": "6daca0c3d14b09de5695cd83cc3e86418f39d8afff1f4116611c0ed7cf5d6937", "baseline_signature": "729209d72e1ee50796f4964f087eb66b7e4fb87a6200e926cf8ef838af92272e", "candidate_features": ["F4Z__basis__basis_change_percentile__12h__value", "F4Z__basis__basis_change_percentile__15m__value", "F4Z__basis__basis_change_percentile__1h__value", "F4Z__basis__basis_change_percentile__4h__value", "F4Z__basis__basis_change_percentile__5m__value", "F4Z__basis__basis_change_rz__12h__value", "F4Z__basis__basis_change_rz__15m__value", "F4Z__basis__basis_change_rz__1h__value", "F4Z__basis__basis_change_rz__4h__value", "F4Z__basis__basis_change_rz__5m__value", "F4Z__basis__basis_level_percentile__12h__value", "F4Z__basis__basis_level_percentile__15m__value", "F4Z__basis__basis_level_percentile__1h__value", "F4Z__basis__basis_level_percentile__4h__value", "F4Z__basis__basis_level_percentile__5m__value", "F4Z__basis__basis_level_rz__12h__value", "F4Z__basis__basis_level_rz__15m__value", "F4Z__basis__basis_level_rz__1h__value", "F4Z__basis__basis_level_rz__4h__value", "F4Z__basis__basis_level_rz__5m__value", "F4Z__basis__basis_path_retention_percentile__12h__value", "F4Z__basis__basis_path_retention_percentile__15m__value", "F4Z__basis__basis_path_retention_percentile__1h__value", "F4Z__basis__basis_path_retention_percentile__4h__value", "F4Z__basis__basis_path_retention_percentile__5m__value", "F4Z__basis__basis_persistence_percentile__12h__value", "F4Z__basis__basis_persistence_percentile__15m__value", "F4Z__basis__basis_persistence_percentile__1h__value", "F4Z__basis__basis_persistence_percentile__4h__value", "F4Z__basis__basis_persistence_percentile__5m__value", "F4Z__basis__basis_range_percentile__12h__value", "F4Z__basis__basis_range_percentile__15m__value", "F4Z__basis__basis_range_percentile__1h__value", "F4Z__basis__basis_range_percentile__4h__value", "F4Z__basis__basis_range_percentile__5m__value", "F4Z__basis__basis_range_rz__12h__value", "F4Z__basis__basis_range_rz__15m__value", "F4Z__basis__basis_range_rz__1h__value", "F4Z__basis__basis_range_rz__4h__value", "F4Z__basis__basis_range_rz__5m__value", "F4Z__flow__cvd_path_efficiency_percentile__12h__value", "F4Z__flow__cvd_path_efficiency_percentile__15m__value", "F4Z__flow__cvd_path_efficiency_percentile__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value", "F4Z__flow__cvd_path_efficiency_percentile__5m__value", "F4Z__flow__flow_acceleration_percentile__15m__12h__value", "F4Z__flow__flow_acceleration_percentile__15m__1h__value", "F4Z__flow__flow_acceleration_percentile__15m__4h__value", "F4Z__flow__flow_acceleration_percentile__1h__12h__value", "F4Z__flow__flow_acceleration_percentile__1h__4h__value", "F4Z__flow__flow_acceleration_percentile__4h__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__15m__value", "F4Z__flow__flow_acceleration_percentile__5m__1h__value", "F4Z__flow__flow_acceleration_percentile__5m__4h__value", "F4Z__flow__flow_acceleration_rz__15m__12h__value", "F4Z__flow__flow_acceleration_rz__15m__1h__value", "F4Z__flow__flow_acceleration_rz__15m__4h__value", "F4Z__flow__flow_acceleration_rz__1h__12h__value", "F4Z__flow__flow_acceleration_rz__1h__4h__value", "F4Z__flow__flow_acceleration_rz__4h__12h__value", "F4Z__flow__flow_acceleration_rz__5m__12h__value", "F4Z__flow__flow_acceleration_rz__5m__15m__value", "F4Z__flow__flow_acceleration_rz__5m__1h__value", "F4Z__flow__flow_acceleration_rz__5m__4h__value", "F4Z__flow__flow_activity_percentile__12h__value", "F4Z__flow__flow_activity_percentile__15m__value", "F4Z__flow__flow_activity_percentile__1h__value", "F4Z__flow__flow_activity_percentile__4h__value", "F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_activity_rz__12h__value", "F4Z__flow__flow_activity_rz__15m__value", "F4Z__flow__flow_activity_rz__1h__value", "F4Z__flow__flow_activity_rz__4h__value", "F4Z__flow__flow_activity_rz__5m__value", "F4Z__flow__flow_confirmation_percentile__12h__value", "F4Z__flow__flow_confirmation_percentile__15m__value", "F4Z__flow__flow_confirmation_percentile__1h__value", "F4Z__flow__flow_confirmation_percentile__4h__value", "F4Z__flow__flow_confirmation_percentile__5m__value", "F4Z__flow__flow_confirmation_rz__12h__value", "F4Z__flow__flow_confirmation_rz__15m__value", "F4Z__flow__flow_confirmation_rz__1h__value", "F4Z__flow__flow_confirmation_rz__4h__value", "F4Z__flow__flow_confirmation_rz__5m__value", "F4Z__flow__flow_cvd_slope_percentile__12h__value", "F4Z__flow__flow_cvd_slope_percentile__15m__value", "F4Z__flow__flow_cvd_slope_percentile__1h__value", "F4Z__flow__flow_cvd_slope_percentile__4h__value", "F4Z__flow__flow_cvd_slope_percentile__5m__value", "F4Z__flow__flow_cvd_slope_rz__12h__value", "F4Z__flow__flow_cvd_slope_rz__15m__value", "F4Z__flow__flow_cvd_slope_rz__1h__value", "F4Z__flow__flow_cvd_slope_rz__4h__value", "F4Z__flow__flow_cvd_slope_rz__5m__value", "F4Z__flow__flow_pressure_percentile__12h__value", "F4Z__flow__flow_pressure_percentile__15m__value", "F4Z__flow__flow_pressure_percentile__1h__value", "F4Z__flow__flow_pressure_percentile__4h__value", "F4Z__flow__flow_pressure_percentile__5m__value", "F4Z__flow__flow_pressure_rz__12h__value", "F4Z__flow__flow_pressure_rz__15m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__flow_pressure_rz__4h__value", "F4Z__flow__flow_pressure_rz__5m__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__4h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__15m__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__4h__value", "F4Z__flow__signed_flow_acceleration_rz__4h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__15m__value", "F4Z__flow__signed_flow_acceleration_rz__5m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__4h__value", "F4Z__funding__funding_change_percentile__15m__value", "F4Z__funding__funding_change_percentile__1h__value", "F4Z__funding__funding_change_percentile__4h__value", "F4Z__funding__funding_change_percentile__5m__value", "F4Z__funding__funding_change_rz__15m__value", "F4Z__funding__funding_change_rz__1h__value", "F4Z__funding__funding_change_rz__4h__value", "F4Z__funding__funding_change_rz__5m__value", "F4Z__funding__funding_level_percentile__15m__value", "F4Z__funding__funding_level_percentile__1h__value", "F4Z__funding__funding_level_percentile__4h__value", "F4Z__funding__funding_level_percentile__5m__value", "F4Z__funding__funding_level_rz__15m__value", "F4Z__funding__funding_level_rz__1h__value", "F4Z__funding__funding_level_rz__4h__value", "F4Z__funding__funding_level_rz__5m__value", "F4Z__funding__funding_path_retention_percentile__15m__value", "F4Z__funding__funding_path_retention_percentile__1h__value", "F4Z__funding__funding_path_retention_percentile__4h__value", "F4Z__funding__funding_path_retention_percentile__5m__value", "F4Z__funding__funding_persistence_percentile__15m__value", "F4Z__funding__funding_persistence_percentile__1h__value", "F4Z__funding__funding_persistence_percentile__4h__value", "F4Z__funding__funding_persistence_percentile__5m__value", "F4Z__funding__funding_range_percentile__15m__value", "F4Z__funding__funding_range_percentile__1h__value", "F4Z__funding__funding_range_percentile__4h__value", "F4Z__funding__funding_range_percentile__5m__value", "F4Z__funding__funding_range_rz__15m__value", "F4Z__funding__funding_range_rz__1h__value", "F4Z__funding__funding_range_rz__4h__value", "F4Z__funding__funding_range_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__5m__value", "F4Z__liquidation__liq_acceleration_percentile__15m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__4h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__4h__value", "F4Z__liquidation__liq_acceleration_percentile__4h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__15m__value", "F4Z__liquidation__liq_acceleration_percentile__5m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__4h__value", "F4Z__liquidation__liq_acceleration_rz__15m__12h__value", "F4Z__liquidation__liq_acceleration_rz__15m__1h__value", "F4Z__liquidation__liq_acceleration_rz__15m__4h__value", "F4Z__liquidation__liq_acceleration_rz__1h__12h__value", "F4Z__liquidation__liq_acceleration_rz__1h__4h__value", "F4Z__liquidation__liq_acceleration_rz__4h__12h__value", "F4Z__liquidation__liq_acceleration_rz__5m__15m__value", "F4Z__liquidation__liq_acceleration_rz__5m__1h__value", "F4Z__liquidation__liq_acceleration_rz__5m__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__5m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__5m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__15m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__1h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__4h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__5m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__12h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__15m__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__1h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_percentile__12h__value", "F4Z__liquidation__liq_side_pressure_percentile__15m__value", "F4Z__liquidation__liq_side_pressure_percentile__1h__value", "F4Z__liquidation__liq_side_pressure_percentile__4h__value", "F4Z__liquidation__liq_side_pressure_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_rz__12h__value", "F4Z__liquidation__liq_side_pressure_rz__15m__value", "F4Z__liquidation__liq_side_pressure_rz__1h__value", "F4Z__liquidation__liq_side_pressure_rz__4h__value", "F4Z__liquidation__liq_side_pressure_rz__5m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__12h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__15m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__1h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__4h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__5m__value", "F4Z__liquidation__venue_breadth_percentile__12h__value", "F4Z__liquidation__venue_breadth_percentile__15m__value", "F4Z__liquidation__venue_breadth_percentile__1h__value", "F4Z__liquidation__venue_breadth_percentile__4h__value", "F4Z__liquidation__venue_breadth_percentile__5m__value", "F4Z__liquidation__venue_fragmentation_percentile__12h__value", "F4Z__liquidation__venue_fragmentation_percentile__15m__value", "F4Z__liquidation__venue_fragmentation_percentile__1h__value", "F4Z__liquidation__venue_fragmentation_percentile__4h__value", "F4Z__liquidation__venue_fragmentation_percentile__5m__value", "F4Z__oi__leverage_unwind_percentile__12h__value", "F4Z__oi__leverage_unwind_percentile__15m__value", "F4Z__oi__leverage_unwind_percentile__1h__value", "F4Z__oi__leverage_unwind_percentile__4h__value", "F4Z__oi__leverage_unwind_percentile__5m__value", "F4Z__oi__leverage_unwind_rz__12h__value", "F4Z__oi__leverage_unwind_rz__15m__value", "F4Z__oi__leverage_unwind_rz__1h__value", "F4Z__oi__leverage_unwind_rz__4h__value", "F4Z__oi__leverage_unwind_rz__5m__value", "F4Z__oi__oi_extreme_recency_percentile__12h__value", "F4Z__oi__oi_extreme_recency_percentile__15m__value", "F4Z__oi__oi_extreme_recency_percentile__1h__value", "F4Z__oi__oi_extreme_recency_percentile__4h__value", "F4Z__oi__oi_extreme_recency_percentile__5m__value", "F4Z__oi__oi_path_retention_percentile__12h__value", "F4Z__oi__oi_path_retention_percentile__15m__value", "F4Z__oi__oi_path_retention_percentile__1h__value", "F4Z__oi__oi_path_retention_percentile__4h__value", "F4Z__oi__oi_path_retention_percentile__5m__value", "F4Z__oi__oi_range_position_percentile__12h__value", "F4Z__oi__oi_range_position_percentile__15m__value", "F4Z__oi__oi_range_position_percentile__1h__value", "F4Z__oi__oi_range_position_percentile__4h__value", "F4Z__oi__oi_range_position_percentile__5m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__12h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__15m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__1h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__4h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__5m__value", "F4Z__price__price_acceleration_percentile__15m__12h__value", "F4Z__price__price_acceleration_percentile__15m__1h__value", "F4Z__price__price_acceleration_percentile__15m__4h__value", "F4Z__price__price_acceleration_percentile__1h__12h__value", "F4Z__price__price_acceleration_percentile__1h__4h__value", "F4Z__price__price_acceleration_percentile__4h__12h__value", "F4Z__price__price_acceleration_percentile__5m__12h__value", "F4Z__price__price_acceleration_percentile__5m__15m__value", "F4Z__price__price_acceleration_percentile__5m__1h__value", "F4Z__price__price_acceleration_percentile__5m__4h__value", "F4Z__price__price_acceleration_rz__15m__12h__value", "F4Z__price__price_acceleration_rz__15m__1h__value", "F4Z__price__price_acceleration_rz__15m__4h__value", "F4Z__price__price_acceleration_rz__1h__12h__value", "F4Z__price__price_acceleration_rz__1h__4h__value", "F4Z__price__price_acceleration_rz__4h__12h__value", "F4Z__price__price_acceleration_rz__5m__12h__value", "F4Z__price__price_acceleration_rz__5m__15m__value", "F4Z__price__price_acceleration_rz__5m__1h__value", "F4Z__price__price_acceleration_rz__5m__4h__value", "F4Z__price__price_displacement_percentile__12h__value", "F4Z__price__price_displacement_percentile__15m__value", "F4Z__price__price_displacement_percentile__1h__value", "F4Z__price__price_displacement_percentile__4h__value", "F4Z__price__price_displacement_percentile__5m__value", "F4Z__price__price_displacement_rz__12h__value", "F4Z__price__price_displacement_rz__15m__value", "F4Z__price__price_displacement_rz__1h__value", "F4Z__price__price_displacement_rz__4h__value", "F4Z__price__price_displacement_rz__5m__value", "F4Z__price__price_excursion_range_percentile__12h__value", "F4Z__price__price_excursion_range_percentile__15m__value", "F4Z__price__price_excursion_range_percentile__1h__value", "F4Z__price__price_excursion_range_percentile__4h__value", "F4Z__price__price_excursion_range_percentile__5m__value", "F4Z__price__price_excursion_range_rz__12h__value", "F4Z__price__price_excursion_range_rz__15m__value", "F4Z__price__price_excursion_range_rz__1h__value", "F4Z__price__price_excursion_range_rz__4h__value", "F4Z__price__price_excursion_range_rz__5m__value", "F4Z__price__price_move_retention_percentile__12h__value", "F4Z__price__price_move_retention_percentile__15m__value", "F4Z__price__price_move_retention_percentile__1h__value", "F4Z__price__price_move_retention_percentile__4h__value", "F4Z__price__price_move_retention_percentile__5m__value", "F4Z__volatility__range_expansion_percentile__15m__12h__value", "F4Z__volatility__range_expansion_percentile__15m__1h__value", "F4Z__volatility__range_expansion_percentile__15m__4h__value", "F4Z__volatility__range_expansion_percentile__1h__12h__value", "F4Z__volatility__range_expansion_percentile__1h__4h__value", "F4Z__volatility__range_expansion_percentile__4h__12h__value", "F4Z__volatility__range_expansion_percentile__5m__12h__value", "F4Z__volatility__range_expansion_percentile__5m__15m__value", "F4Z__volatility__range_expansion_percentile__5m__1h__value", "F4Z__volatility__range_expansion_percentile__5m__4h__value", "F4Z__volatility__range_expansion_rz__15m__12h__value", "F4Z__volatility__range_expansion_rz__15m__1h__value", "F4Z__volatility__range_expansion_rz__15m__4h__value", "F4Z__volatility__range_expansion_rz__1h__12h__value", "F4Z__volatility__range_expansion_rz__1h__4h__value", "F4Z__volatility__range_expansion_rz__4h__12h__value", "F4Z__volatility__range_expansion_rz__5m__12h__value", "F4Z__volatility__range_expansion_rz__5m__15m__value", "F4Z__volatility__range_expansion_rz__5m__1h__value", "F4Z__volatility__range_expansion_rz__5m__4h__value", "F4Z__volatility__realized_variance_level_percentile__12h__value", "F4Z__volatility__realized_variance_level_percentile__15m__value", "F4Z__volatility__realized_variance_level_percentile__1h__value", "F4Z__volatility__realized_variance_level_percentile__4h__value", "F4Z__volatility__realized_variance_level_percentile__5m__value", "F4Z__volatility__realized_variance_level_rz__12h__value", "F4Z__volatility__realized_variance_level_rz__15m__value", "F4Z__volatility__realized_variance_level_rz__1h__value", "F4Z__volatility__realized_variance_level_rz__4h__value", "F4Z__volatility__realized_variance_level_rz__5m__value", "F4Z__volatility__vol_acceleration_percentile__15m__12h__value", "F4Z__volatility__vol_acceleration_percentile__15m__1h__value", "F4Z__volatility__vol_acceleration_percentile__15m__4h__value", "F4Z__volatility__vol_acceleration_percentile__1h__12h__value", "F4Z__volatility__vol_acceleration_percentile__1h__4h__value", "F4Z__volatility__vol_acceleration_percentile__4h__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__15m__value", "F4Z__volatility__vol_acceleration_percentile__5m__1h__value", "F4Z__volatility__vol_acceleration_percentile__5m__4h__value", "F4Z__volatility__vol_acceleration_rz__15m__12h__value", "F4Z__volatility__vol_acceleration_rz__15m__1h__value", "F4Z__volatility__vol_acceleration_rz__15m__4h__value", "F4Z__volatility__vol_acceleration_rz__1h__12h__value", "F4Z__volatility__vol_acceleration_rz__1h__4h__value", "F4Z__volatility__vol_acceleration_rz__4h__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__15m__value", "F4Z__volatility__vol_acceleration_rz__5m__1h__value", "F4Z__volatility__vol_acceleration_rz__5m__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__12h__value", "F4Z__volatility__vol_shock_concentration_percentile__15m__value", "F4Z__volatility__vol_shock_concentration_percentile__1h__value", "F4Z__volatility__vol_shock_concentration_percentile__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__5m__value", "F4Z__volatility__vol_shock_recency_percentile__12h__value", "F4Z__volatility__vol_shock_recency_percentile__15m__value", "F4Z__volatility__vol_shock_recency_percentile__1h__value", "F4Z__volatility__vol_shock_recency_percentile__4h__value", "F4Z__volatility__vol_shock_recency_percentile__5m__value"], "pilot_features": ["F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value"], "pilot_horizons": [15, 2880, 18000], "horizons": [15, 60, 240, 480, 960, 2880, 4320, 7200, 12000, 18000], "months": ["2022-04", "2022-05", "2022-06", "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12", "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06", "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"], "origin_end_exclusive": "2024-07-01T00:00:00Z", "reserve_start": "2024-07-24T00:01:00Z", "cadence_minutes": 30, "folds": [["2023-04-01", "2023-07-01"], ["2023-07-01", "2023-10-01"], ["2023-10-01", "2024-01-01"], ["2024-01-01", "2024-04-01"], ["2024-04-01", "2024-07-01"]], "grid": [[2, 64], [2, 256], [3, 64], [3, 256], [5, 64], [5, 256]], "minimum_observed_training": 1000, "inner_validation": "Preceding three calendar months; inner training label_end < inner start, inner validation label_end < outer start. Outer training label_end < outer start.", "model": "Single raw F4Z input plus explicit missing indicator; DecisionTreeRegressor absolute_error, seed48, depth/leaf grid only.", "preprocessing": "Train-only finite-value median imputation plus missing indicator. No feature scaling. Baseline_n >=1000, finite, valid, recorded available_after <= origin, scientific_status PROVISIONAL explicitly allowed for provisional research only.", "selection": "Lowest inner MAE; ties shallower depth then larger leaf. Never use outer MAE to choose fold settings.", "ineligible_policy": "If fewer than1000 observed training values or constant observed training feature: no fit; if no eligible inner setting or outer training insufficient, predict outer-training target median on all outer rows. Report fallback and coverage.", "evaluation": "Exact original baseline rows; primary MAE minutes, secondary percent of horizon; pooled row-weighted errors; compare all three baselines. Record missing/observed subset errors. No significance/global optimum/trading claim.", "pilot_scope": "3 flow features x3 horizons x5 folds x(6 inner+1 outer) = at most315 fits. Features selected for differing market meanings and sampled coverage, not outcome scores.", "pilot_meanings": {"F4Z__flow__flow_activity_percentile__5m__value": "Recent flow activity relative to history", "F4Z__flow__flow_pressure_rz__1h__value": "Directional trading pressure relative to history", "F4Z__flow__cvd_path_efficiency_percentile__4h__value": "How consistently cumulative trading flow follows one direction"}, "cache_policy": "One compact research-only NPZ per horizon/family in Drive, no PC cache. Float64 values preserve source conversion. Checkpoint each month; immutable signature/hash checks. Family cache includes all candidate columns from that family (85 flow) to avoid rereading later. <=128MiB per cache.", "checkpoint_policy": "Per-fit compact JSON scores in Timing_Screen_Fit_Scores; no model binaries or row predictions. Partial family/horizon data resume completed months. An interrupted unsaved fit may restart. Pilot completed fits reused by full mode under identical identity/versions.", "workers": "Up to4 bounded I/O workers; up to2 fit workers; native thread pools limited. No automatic full run after pilot.", "limitations": ["F4Z upstream timing/normalization remains unresolved; PROVISIONAL research only.", "Sample-based candidate preselection does not prove excluded features useless across full history.", "Single-feature trees do not establish globally best features or combinations.", "Reserved period previously exposed in other project work but excluded from this task.", "Errors overlap in time; no iid significance assumptions."], "full_launch_policy": "Review pilot correctness, runtime, memory and results before authorizing full386-feature expansion. Do not change to full mode yet."}, "sources": {"X": {"id": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "name": "Handoff 1_F4_f4Z.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "043b2f570cfb63215749385d72c7fe59", "size": "15600742757"}, "Y": {"id": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM", "name": "HANDOFF_2_CANONICAL_XY_V1_1.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "bf2cb8dd274222747fde1b2e979b42b7", "size": "3033662926"}}, "runner_sha256": "25ae53d6a34e1695381372f46bcd26298428c341517336f3096e6d9815b172f0", "versions": {"python": "3.13.18", "numpy": "2.4.6", "pandas": "2.2.3", "sklearn": "1.6.1", "pyarrow": "23.0.1"}}')
RUNNER["run_1h"](config,pilot_identity,hashes,session_hours=4)